# Nepali Fact-Check Dataset Scraper

Sequential, per-site scraping pipeline. Output: **one CSV only**, written next to this notebook (no tables).

Site findings from manual inspection (Nov 2026):

| Site | Discovery method | Notes |
| --- | --- | --- |
| Nepal Factcheck | WordPress REST API (`/wp-json/wp/v2/posts`), category `तथ्यजाँच` | Verdict = child categories (`मिथ्या सूचना`, `भ्रामक सूचना`, ...). Claim = section under `के हो दाबी ?` heading |
| The Ruju | n/a | HTTP 403 on every URL incl. robots.txt -> logged and skipped (no bypass) |
| TechPana Fact Check | Static JS bundle of a React SPA | Only title / verdict / author / category / date / image are published; no per-article URL or body |
| Karnali Factcheck | WordPress REST API | Same CMS family as Nepal Factcheck |
| Lumbini Factcheck | WordPress REST API | Same CMS family as Nepal Factcheck |
| Sachilo | `sitemap.xml` (`/v/` pages) | schema.org `ClaimReview` JSON-LD, one page may hold several claims |
| NepalCheck | n/a | robots.txt `Disallow: /` -> skipped |
| South Asia Check | n/a | Domain now serves unrelated casino spam -> skipped |


In [0]:
%python
# ============================================================
# STEP 1 — Imports, configuration, logging and HTTP session
# ============================================================
import csv
import html
import json
import logging
import os
import re
import sys
import time
import urllib.robotparser
from collections import Counter
from datetime import datetime, timezone
from urllib.parse import urljoin, urlparse, urlunparse

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ---------------- Configuration (edit here) ----------------
USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
)
REQUEST_TIMEOUT = 30          # seconds per request
REQUEST_DELAY = 1.5           # polite delay between requests (seconds)
MAX_RETRIES = 4               # retries for 429/5xx/connection errors
BACKOFF_FACTOR = 2            # exponential backoff: 2, 4, 8, 16 s
MAX_LISTING_PAGES = 30        # safety limit for pagination per site
MAX_ARTICLES_PER_SITE = 3000  # safety limit for articles per site
VALIDATE_HTML_SAMPLES = 3     # WP sites: compare API metadata to visible HTML for N posts
CSV_FILENAME = "nepali_fact_check_dataset.csv"
FAILURES_CSV_FILENAME = "nepali_fact_check_failures.csv"

OUTPUT_COLUMNS = [
    "source", "article_url", "article_id", "title", "claim", "verdict",
    "claimant", "claim_date", "fact_check_date", "published_date", "author",
    "category", "language", "description", "article_text", "evidence",
    "sources", "correction", "image_url", "tags", "scraped_at",
]

# ---------------- Logging ----------------
logger = logging.getLogger("nepali_factcheck_scraper")
logger.setLevel(logging.INFO)
logger.propagate = False
if not logger.handlers:
    log_handler = logging.StreamHandler(sys.stdout)
    log_handler.setFormatter(
        logging.Formatter("%(asctime)s | %(levelname)s | %(message)s", "%Y-%m-%d %H:%M:%S")
    )
    logger.addHandler(log_handler)

# Surface urllib3 retry attempts ("Incremented Retry ..." / "Retrying ...") in the same log
for urllib3_logger_name, urllib3_level in [("urllib3.util.retry", logging.DEBUG),
                                           ("urllib3.connectionpool", logging.WARNING)]:
    urllib3_logger = logging.getLogger(urllib3_logger_name)
    urllib3_logger.setLevel(urllib3_level)
    urllib3_logger.propagate = False
    if not urllib3_logger.handlers:
        urllib3_logger.addHandler(logger.handlers[0])

# ---------------- HTTP session with retries + exponential backoff ----------------
retry_config = Retry(
    total=MAX_RETRIES,
    connect=MAX_RETRIES,
    read=MAX_RETRIES,
    status=MAX_RETRIES,
    backoff_factor=BACKOFF_FACTOR,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET", "HEAD"],
    respect_retry_after_header=True,
    raise_on_status=False,   # return the final response so the status code can be logged
)
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=retry_config))
session.mount("http://", HTTPAdapter(max_retries=retry_config))
session.headers.update({
    "User-Agent": USER_AGENT,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,application/json;q=0.8,*/*;q=0.7",
    "Accept-Language": "ne-NP,ne;q=0.9,en;q=0.8",
})

# ---------------- Rerun-safe in-memory containers ----------------
# Keyed by source so re-running one site's cell replaces (not duplicates) its results.
records_by_source = {}
failures_by_source = {}
stats_by_source = {}

RUN_STARTED_AT = datetime.now(timezone.utc)
logger.info("Configuration loaded | delay=%.1fs timeout=%ss retries=%s backoff=%s",
            REQUEST_DELAY, REQUEST_TIMEOUT, MAX_RETRIES, BACKOFF_FACTOR)
logger.info("pandas=%s requests=%s", pd.__version__, requests.__version__)

2026-11-22 15:07:16 | INFO | Configuration loaded | delay=1.5s timeout=30s retries=4 backoff=2
2026-11-22 15:07:16 | INFO | pandas=2.3.3 requests=2.34.2


In [0]:
%python
# ============================================================
# STEP 2 — Determine the notebook's workspace path programmatically
# ============================================================
notebook_context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
NOTEBOOK_PATH = notebook_context.notebookPath().get()
NOTEBOOK_DIR = os.path.dirname(NOTEBOOK_PATH)
logger.info("Notebook path: %s", NOTEBOOK_PATH)
logger.info("Notebook directory (workspace view): %s", NOTEBOOK_DIR)

2026-11-22 15:07:20 | INFO | Notebook path: /Users/ashresth.ycotek@cpaxtra.co.th/Testss/New Notebook 2026-11-22 20:42:31
2026-11-22 15:07:20 | INFO | Notebook directory (workspace view): /Users/ashresth.ycotek@cpaxtra.co.th/Testss


In [0]:
%python
# ============================================================
# STEP 3 — Resolve the output directory (same folder as this notebook)
# ============================================================
# Workspace paths are exposed to Python on the driver under /Workspace/...
if NOTEBOOK_DIR.startswith("/Workspace"):
    OUTPUT_DIR = NOTEBOOK_DIR
else:
    OUTPUT_DIR = "/Workspace" + NOTEBOOK_DIR

logger.info("Resolved output directory (filesystem path): %s", OUTPUT_DIR)
if not os.path.isdir(OUTPUT_DIR):
    raise RuntimeError(f"Notebook directory is not visible on the driver filesystem: {OUTPUT_DIR}")
if not os.access(OUTPUT_DIR, os.W_OK):
    raise RuntimeError(f"Notebook directory is not writable: {OUTPUT_DIR}. Not choosing another location silently.")
logger.info("Output directory exists and is writable — CSV will be written here")

2026-11-22 15:07:26 | INFO | Resolved output directory (filesystem path): /Workspace/Users/ashresth.ycotek@cpaxtra.co.th/Testss
2026-11-22 15:07:26 | INFO | Output directory exists and is writable — CSV will be written here


In [0]:
%python
# ============================================================
# STEP 4 — Scrape Nepal Factcheck (WordPress REST API)
# ============================================================
# Inspection findings:
#   * robots.txt only disallows /wp-admin/ -> /wp-json/ is allowed
#   * Fact checks live in category "तथ्यजाँच"; its child categories are the verdict labels
#     (मिथ्या सूचना / भ्रामक सूचना / अपुष्ट सूचना / सही सूचना / एआई जेनेरेटेड)
#   * Explainers (एक्सप्लेनर) are separate -> excluded by the category filter
#   * Article body: h2 "के हो दाबी ?" (claim) -> "के हो तथ्य ?" (evidence) -> "तथ्यजाँच" (conclusion)
#   * Pagination: ?page=N with X-WP-TotalPages response header
SOURCE = "Nepal Factcheck"
BASE_URL = "https://nepalfactcheck.org"
OWN_DOMAINS = {"nepalfactcheck.org", "www.nepalfactcheck.org"}
FACTCHECK_CATEGORY_NAME = "तथ्यजाँच"

site_records, site_failures = [], []
site_stats = {"source": SOURCE, "pages_discovered": 0, "urls_found": 0, "successful": 0,
              "skipped": 0, "failed": 0, "duplicate_urls": 0, "status": "STARTED"}
logger.info("=" * 60)
logger.info("Starting %s scraper", SOURCE)

# ---- 4.1 robots.txt ----
robots = urllib.robotparser.RobotFileParser()
robots_url = BASE_URL + "/robots.txt"
try:
    logger.info("Fetching robots.txt: %s", robots_url)
    resp = session.get(robots_url, timeout=REQUEST_TIMEOUT)
    logger.info("HTTP %s", resp.status_code)
    if resp.status_code in (401, 403):
        robots.disallow_all = True
    elif resp.status_code >= 400:
        robots.allow_all = True
    else:
        robots.parse(resp.text.splitlines())
except requests.RequestException as exc:
    logger.error("robots.txt request failed: %s -> treating site as disallowed", exc)
    robots.disallow_all = True
time.sleep(REQUEST_DELAY)

posts_api = BASE_URL + "/wp-json/wp/v2/posts"
if not robots.can_fetch(USER_AGENT, posts_api):
    logger.warning("robots.txt disallows %s — skipping %s", posts_api, SOURCE)
    site_stats["status"] = "BLOCKED_BY_ROBOTS"
else:
    # ---- 4.2 Category map (verdict labels are child categories of तथ्यजाँच) ----
    categories = {}
    cat_url = BASE_URL + "/wp-json/wp/v2/categories"
    try:
        logger.info("Fetching categories: %s", cat_url)
        resp = session.get(cat_url, params={"per_page": 100}, timeout=REQUEST_TIMEOUT)
        logger.info("HTTP %s", resp.status_code)
        resp.raise_for_status()
        for cat in resp.json():
            categories[cat["id"]] = cat
    except (requests.RequestException, ValueError) as exc:
        logger.error("Could not load categories: %s", exc)
        site_failures.append({"source": SOURCE, "url": cat_url,
                              "status_code": getattr(getattr(exc, "response", None), "status_code", None),
                              "error": str(exc)})
    time.sleep(REQUEST_DELAY)

    factcheck_cat_ids = [cid for cid, cat in categories.items()
                         if html.unescape(cat["name"]).strip() == FACTCHECK_CATEGORY_NAME]
    verdict_cat_ids = {cid for cid, cat in categories.items() if cat.get("parent") in factcheck_cat_ids}
    logger.info("Fact-check category ids: %s | verdict categories: %s", factcheck_cat_ids,
                [categories[c]["name"] for c in verdict_cat_ids])
    if not factcheck_cat_ids:
        logger.error("Fact-check category '%s' not found — cannot filter fact checks", FACTCHECK_CATEGORY_NAME)
        site_stats["status"] = "FAILED_NO_CATEGORY"

    # ---- 4.3 Sitemap cross-check (informational only) ----
    try:
        sm_url = BASE_URL + "/sitemap_index.xml"
        logger.info("Fetching sitemap index for cross-check: %s", sm_url)
        resp = session.get(sm_url, timeout=REQUEST_TIMEOUT)
        logger.info("HTTP %s", resp.status_code)
        post_sitemaps = [u for u in re.findall(r"<loc>(.*?)</loc>", resp.text) if "post-sitemap" in u]
        sitemap_post_urls = set()
        for sm in post_sitemaps:
            time.sleep(REQUEST_DELAY)
            r_sm = session.get(sm, timeout=REQUEST_TIMEOUT)
            logger.info("Fetching sitemap: %s | HTTP %s", sm, r_sm.status_code)
            sitemap_post_urls.update(re.findall(r"<loc>(.*?)</loc>", r_sm.text))
        logger.info("Sitemap lists %d post URLs (all post types incl. explainers)", len(sitemap_post_urls))
    except requests.RequestException as exc:
        logger.warning("Sitemap cross-check failed: %s", exc)
    time.sleep(REQUEST_DELAY)

    # ---- 4.4 Paginate the posts API, filtered to fact-check category ----
    raw_posts = []
    page = 1
    total_pages = None
    while factcheck_cat_ids and page <= MAX_LISTING_PAGES:
        params = {"per_page": 100, "page": page, "categories": ",".join(map(str, factcheck_cat_ids)),
                  "_embed": "author,wp:term,wp:featuredmedia", "orderby": "date", "order": "desc"}
        try:
            logger.info("Fetching page %d: %s", page, posts_api)
            resp = session.get(posts_api, params=params, timeout=REQUEST_TIMEOUT)
            logger.info("HTTP %s", resp.status_code)
            if resp.status_code == 400 and page > 1:
                logger.info("API reports no more pages (HTTP 400 rest_post_invalid_page_number)")
                break
            resp.raise_for_status()
            page_posts = resp.json()
        except (requests.RequestException, ValueError) as exc:
            logger.error("Listing page %d failed: %s", page, exc)
            site_failures.append({"source": SOURCE, "url": f"{posts_api}?page={page}",
                                  "status_code": getattr(getattr(exc, "response", None), "status_code", None),
                                  "error": str(exc)})
            break
        if total_pages is None:
            total_pages = int(resp.headers.get("X-WP-TotalPages", 1))
            logger.info("API reports %s fact-check posts across %d pages",
                        resp.headers.get("X-WP-Total"), total_pages)
        if not page_posts:
            logger.info("Page %d returned no posts — stopping pagination", page)
            break
        raw_posts.extend(page_posts)
        site_stats["pages_discovered"] = page
        logger.info("Page %d: %d posts (running total %d)", page, len(page_posts), len(raw_posts))
        time.sleep(REQUEST_DELAY)
        if page >= total_pages or len(raw_posts) >= MAX_ARTICLES_PER_SITE:
            break
        page += 1

    # ---- 4.5 Deduplicate URLs ----
    seen_urls, unique_posts = set(), []
    for post in raw_posts:
        p_url = urlunparse(urlparse(post.get("link", ""))._replace(query="", fragment="")).rstrip("/").lower()
        if p_url in seen_urls:
            site_stats["duplicate_urls"] += 1
            logger.warning("Duplicate URL skipped: %s", post.get("link"))
            continue
        seen_urls.add(p_url)
        unique_posts.append(post)
    unique_posts = unique_posts[:MAX_ARTICLES_PER_SITE]
    site_stats["urls_found"] = len(unique_posts)
    logger.info("Found %d unique article URLs", len(unique_posts))

    # ---- 4.6 Parse each post ----
    for idx, post in enumerate(unique_posts):
        article_url = post.get("link")
        try:
            soup = BeautifulSoup(post["content"]["rendered"], "html.parser")
            for junk in soup.select("script, style, noscript, iframe, form, video, audio, .sharedaddy, "
                                    ".jp-relatedposts, .wp-video, .addtoany_share_save_container"):
                junk.decompose()

            # Author: embedded WP user, else a byline list (<ul><li><strong>Name</strong>) at the top
            author = None
            for emb_author in post.get("_embedded", {}).get("author", []):
                if emb_author.get("name"):
                    author = emb_author["name"].strip()
            first_el = soup.find(True)
            if first_el is not None and first_el.name == "ul" and first_el.find("strong") \
                    and len(first_el.get_text(strip=True)) < 60:
                if author is None:
                    author = first_el.get_text(" ", strip=True)
                first_el.decompose()

            # External sources (exclude own/network domains and uploaded media)
            source_links = []
            for a_tag in soup.find_all("a", href=True):
                href = a_tag["href"].strip()
                host = urlparse(href).netloc.lower()
                if href.startswith("http") and host and host not in OWN_DOMAINS \
                        and "nepalfactcheck.org" not in host and href not in source_links:
                    source_links.append(href)

            # Walk the content blocks, grouping paragraphs under their headings
            sections, all_blocks, seen_blocks = [], [], set()
            current_heading, current_texts = None, []
            for el in soup.find_all(["h2", "h3", "h4", "p", "li", "blockquote"]):
                txt = re.sub(r"\s+", " ", el.get_text(" ", strip=True)).strip()
                if not txt or txt in seen_blocks:
                    continue
                seen_blocks.add(txt)
                all_blocks.append(txt)
                # Many posts use a bold lead-in inside a paragraph as the heading, e.g.
                #   <p><strong>के हो दाबी ?</strong><br/> claim text ...</p>
                # -> heading = bold text, remainder of the paragraph = first line of that section
                bold_el = el.find(["strong", "b"]) if el.name == "p" else None
                bold_text = re.sub(r"\s+", " ", bold_el.get_text(" ", strip=True)).strip() if bold_el else ""
                is_pseudo_heading = (
                    bold_text != "" and len(bold_text) < 60 and txt.startswith(bold_text)
                    and re.search(r"दाबी|दावी|तथ्य", bold_text) is not None
                )
                if el.name in ("h2", "h3", "h4"):
                    sections.append((current_heading, current_texts))
                    current_heading, current_texts = txt, []
                elif is_pseudo_heading:
                    sections.append((current_heading, current_texts))
                    remainder = txt[len(bold_text):].strip()
                    current_heading, current_texts = bold_text, ([remainder] if remainder else [])
                else:
                    current_texts.append(txt)
            sections.append((current_heading, current_texts))

            # Claim: section under a heading mentioning दाबी/दावी; else first headed section before any तथ्य heading
            claim = None
            for heading, texts in sections:
                if heading and re.search(r"दाबी|दावी", heading) and texts:
                    claim = "\n".join(texts)
                    break
            if claim is None:
                for heading, texts in sections:
                    if heading and "तथ्य" in heading:
                        break
                    if heading and texts:
                        claim = "\n".join(texts)
                        break

            # Evidence: from the first तथ्य heading onward; the conclusion (तथ्यजाँच) is appended with a label
            evidence_parts, in_evidence = [], False
            for heading, texts in sections:
                if heading and "तथ्य" in heading:
                    in_evidence = True
                if in_evidence and texts:
                    evidence_parts.append(f"{heading}\n" + "\n".join(texts))
            evidence = "\n\n".join(evidence_parts) or None

            correction = None
            for heading, texts in sections:
                if heading and re.search(r"सच्याइ|संशोधन|correction", heading, re.I):
                    correction = "\n".join(texts) or heading

            # Terms: verdict (child categories of तथ्यजाँच), other categories, tags
            verdict_labels, other_categories, tag_names = [], [], []
            for term_group in post.get("_embedded", {}).get("wp:term", []):
                for term in term_group:
                    term_name = html.unescape(term.get("name", "")).strip()
                    if term.get("taxonomy") == "category":
                        if term.get("id") in verdict_cat_ids:
                            verdict_labels.append(term_name)
                        elif term.get("id") not in factcheck_cat_ids:
                            other_categories.append(term_name)
                    elif term.get("taxonomy") == "post_tag":
                        tag_names.append(term_name)

            image_url = post.get("jetpack_featured_media_url") or None
            if not image_url:
                for media in post.get("_embedded", {}).get("wp:featuredmedia", []):
                    image_url = media.get("source_url") or image_url

            article_text = "\n\n".join(all_blocks)
            dev_chars = len(re.findall(r"[\u0900-\u097F]", article_text))
            lat_chars = len(re.findall(r"[A-Za-z]", article_text))
            dev_ratio = dev_chars / max(dev_chars + lat_chars, 1)
            language = "ne" if dev_ratio >= 0.7 else ("en" if dev_ratio <= 0.3 else "mixed")

            title = html.unescape(re.sub(r"<[^>]+>", "", post["title"]["rendered"])).strip()
            description = BeautifulSoup(post.get("excerpt", {}).get("rendered", ""), "html.parser").get_text(" ", strip=True) or None

            # Validate API metadata against the visible HTML page for a few samples
            if idx < VALIDATE_HTML_SAMPLES and robots.can_fetch(USER_AGENT, article_url):
                logger.info("Validating metadata against HTML: %s", article_url)
                page_resp = session.get(article_url, timeout=REQUEST_TIMEOUT)
                logger.info("HTTP %s", page_resp.status_code)
                page_soup = BeautifulSoup(page_resp.text, "html.parser")
                visible_h1 = page_soup.find("h1")
                visible_title = visible_h1.get_text(" ", strip=True) if visible_h1 else None
                if visible_title and visible_title.strip() == title.strip():
                    logger.info("Visible <h1> matches API title")
                else:
                    logger.warning("Visible <h1> '%s' differs from API title '%s'", visible_title, title)
                time.sleep(REQUEST_DELAY)

            record = {
                "source": SOURCE, "article_url": article_url, "article_id": str(post.get("id")),
                "title": title, "claim": claim,
                "verdict": ", ".join(verdict_labels) or None,
                "claimant": None, "claim_date": None, "fact_check_date": None,
                "published_date": post.get("date"), "author": author,
                "category": ", ".join(other_categories) or None, "language": language,
                "description": description, "article_text": article_text or None,
                "evidence": evidence, "sources": " | ".join(source_links) or None,
                "correction": correction, "image_url": image_url,
                "tags": ", ".join(tag_names) or None,
                "scraped_at": datetime.now(timezone.utc).isoformat(),
            }
            missing = [f for f in ("title", "claim", "verdict", "article_text") if not record[f]]
            if missing:
                logger.warning("Missing fields %s | %s", missing, article_url)
            if not record["article_text"]:
                site_stats["skipped"] += 1
                logger.warning("Skipped (empty body): %s", article_url)
                continue
            site_records.append(record)
            site_stats["successful"] += 1
        except Exception as exc:  # recoverable: log and continue with next article
            site_stats["failed"] += 1
            logger.exception("Parsing error for %s: %s", article_url, exc)
            site_failures.append({"source": SOURCE, "url": article_url, "status_code": None, "error": repr(exc)})

    if site_stats["status"] == "STARTED":
        site_stats["status"] = "OK"

records_by_source[SOURCE] = site_records
failures_by_source[SOURCE] = site_failures
stats_by_source[SOURCE] = site_stats
logger.info("Successfully scraped %d articles", site_stats["successful"])
if site_stats["failed"]:
    logger.warning("%d articles failed parsing", site_stats["failed"])
logger.info("%s finished | %s", SOURCE, site_stats)

2026-11-22 15:12:35 | INFO | ============================================================
2026-11-22 15:12:35 | INFO | Starting Nepal Factcheck scraper
2026-11-22 15:12:35 | INFO | Fetching robots.txt: https://nepalfactcheck.org/robots.txt
2026-11-22 15:12:35 | INFO | HTTP 200
2026-11-22 15:12:36 | INFO | Fetching categories: https://nepalfactcheck.org/wp-json/wp/v2/categories
2026-11-22 15:12:37 | INFO | HTTP 200
2026-11-22 15:12:38 | INFO | Fact-check category ids: [1] | verdict categories: ['मिथ्या सूचना', 'अपुष्ट सूचना', 'भ्रामक सूचना', 'सही सूचना', 'एआई जेनेरेटेड']
2026-11-22 15:12:38 | INFO | Fetching sitemap index for cross-check: https://nepalfactcheck.org/sitemap_index.xml
2026-11-22 15:12:39 | INFO | HTTP 200
2026-11-22 15:12:41 | INFO | Fetching sitemap: https://nepalfactcheck.org/post-sitemap1.xml | HTTP 200
2026-11-22 15:12:43 | INFO | Fetching sitemap: https://nepalfactcheck.org/post-sitemap2.xml | HTTP 200
2026-11-22 15:12:44 | INFO | Fetching sitemap: https://nepalfactc

In [0]:
%python
# ============================================================
# STEP 5 — Scrape The Ruju
# ============================================================
# Inspection findings: every URL tested (/, /robots.txt, /sitemap.xml, /feed/, /wp-json/)
# returned HTTP 403 from nginx. A 401/403 on robots.txt is treated as "disallow all"
# (same convention as Python's urllib.robotparser). We do NOT attempt to bypass this.
SOURCE = "The Ruju"
BASE_URL = "https://www.theruju.com"

site_records, site_failures = [], []
site_stats = {"source": SOURCE, "pages_discovered": 0, "urls_found": 0, "successful": 0,
              "skipped": 0, "failed": 0, "duplicate_urls": 0, "status": "STARTED"}
logger.info("=" * 60)
logger.info("Starting %s scraper", SOURCE)

# ---- 5.1 robots.txt ----
robots = urllib.robotparser.RobotFileParser()
robots_url = BASE_URL + "/robots.txt"
robots_status = None
try:
    logger.info("Fetching robots.txt: %s", robots_url)
    resp = session.get(robots_url, timeout=REQUEST_TIMEOUT)
    robots_status = resp.status_code
    logger.info("HTTP %s", resp.status_code)
    if resp.status_code in (401, 403):
        robots.disallow_all = True
    elif resp.status_code >= 400:
        robots.allow_all = True
    else:
        robots.parse(resp.text.splitlines())
except requests.RequestException as exc:
    logger.error("robots.txt request failed: %s -> treating site as disallowed", exc)
    robots.disallow_all = True
time.sleep(REQUEST_DELAY)

# ---- 5.2 Homepage (only if allowed) ----
if not robots.can_fetch(USER_AGENT, BASE_URL + "/"):
    logger.warning("Access restricted: robots.txt returned HTTP %s -> treated as disallow-all. "
                   "Not bypassing access restrictions; skipping %s", robots_status, SOURCE)
    site_stats["status"] = f"ACCESS_FORBIDDEN_HTTP_{robots_status}"
    site_failures.append({"source": SOURCE, "url": robots_url, "status_code": robots_status,
                          "error": "Site returns HTTP 403 (access restricted); not scraped"})
else:
    logger.info("Fetching homepage: %s", BASE_URL + "/")
    resp = session.get(BASE_URL + "/", timeout=REQUEST_TIMEOUT)
    logger.info("HTTP %s", resp.status_code)
    if resp.status_code != 200:
        site_stats["status"] = f"HTTP_{resp.status_code}"
        site_failures.append({"source": SOURCE, "url": BASE_URL + "/", "status_code": resp.status_code,
                              "error": "Homepage not retrievable with normal HTTP"})
    else:
        # Site became reachable: selectors must be inspected before scraping (do not guess them)
        logger.warning("%s is now reachable — inspect its HTML and add site-specific selectors here", SOURCE)
        site_stats["status"] = "REACHABLE_NOT_IMPLEMENTED"

records_by_source[SOURCE] = site_records
failures_by_source[SOURCE] = site_failures
stats_by_source[SOURCE] = site_stats
logger.info("%s finished | %s", SOURCE, site_stats)

2026-11-22 15:19:47 | INFO | ============================================================
2026-11-22 15:19:47 | INFO | Starting The Ruju scraper
2026-11-22 15:19:47 | INFO | Fetching robots.txt: https://www.theruju.com/robots.txt
2026-11-22 15:19:48 | INFO | HTTP 403
2026-11-22 15:19:49 | WARNING | Access restricted: robots.txt returned HTTP 403 -> treated as disallow-all. Not bypassing access restrictions; skipping The Ruju
2026-11-22 15:19:49 | INFO | The Ruju finished | {'source': 'The Ruju', 'pages_discovered': 0, 'urls_found': 0, 'successful': 0, 'skipped': 0, 'failed': 0, 'duplicate_urls': 0, 'status': 'ACCESS_FORBIDDEN_HTTP_403'}


In [0]:
%python
# ============================================================
# STEP 6 — Scrape TechPana Fact Check
# ============================================================
# Inspection findings:
#   * The homepage is a React single-page app: raw HTML only contains <div id="root"></div>
#   * No sitemap, RSS or WordPress API (all 404); robots.txt is 404 -> no restrictions
#   * The fact-check list is hard-coded in the site's own static JS bundle as objects:
#       {id:N,title:"...",imageUrl:"...",author:<authors>.<key>,category:"...",verdict:"...",date:"..."}
#     plus an author map  <key>:{name:"...",image:"...",url:"..."}
#   * The bundle has NO per-article URL, claim text or article body -> those fields stay None
SOURCE = "TechPana Fact Check"
BASE_URL = "https://tpfactcheck.com"

site_records, site_failures = [], []
site_stats = {"source": SOURCE, "pages_discovered": 0, "urls_found": 0, "successful": 0,
              "skipped": 0, "failed": 0, "duplicate_urls": 0, "status": "STARTED"}
logger.info("=" * 60)
logger.info("Starting %s scraper", SOURCE)

# ---- 6.1 robots.txt ----
robots = urllib.robotparser.RobotFileParser()
robots_url = BASE_URL + "/robots.txt"
try:
    logger.info("Fetching robots.txt: %s", robots_url)
    resp = session.get(robots_url, timeout=REQUEST_TIMEOUT)
    logger.info("HTTP %s", resp.status_code)
    if resp.status_code in (401, 403):
        robots.disallow_all = True
    elif resp.status_code >= 400:
        robots.allow_all = True
    else:
        robots.parse(resp.text.splitlines())
except requests.RequestException as exc:
    logger.error("robots.txt request failed: %s -> treating site as disallowed", exc)
    robots.disallow_all = True
time.sleep(REQUEST_DELAY)

if not robots.can_fetch(USER_AGENT, BASE_URL + "/"):
    logger.warning("robots.txt disallows %s — skipping", BASE_URL)
    site_stats["status"] = "BLOCKED_BY_ROBOTS"
else:
    # ---- 6.2 Homepage: check whether content is present in raw HTML ----
    bundle_texts = []
    try:
        logger.info("Fetching page: %s", BASE_URL + "/")
        resp = session.get(BASE_URL + "/", timeout=REQUEST_TIMEOUT)
        logger.info("HTTP %s", resp.status_code)
        resp.raise_for_status()
        home_soup = BeautifulSoup(resp.text, "html.parser")
        root_div = home_soup.find(id="root")
        if root_div is not None and not root_div.get_text(strip=True):
            logger.warning("Dynamic rendering required — article list is not in the raw HTML. "
                           "Reading the site's own static JS bundle instead (no browser needed).")
        bundle_urls = [urljoin(BASE_URL + "/", s["src"]) for s in home_soup.find_all("script", src=True)
                       if s.get("type") == "module" and urlparse(urljoin(BASE_URL + "/", s["src"])).netloc == "tpfactcheck.com"]
        logger.info("Found %d first-party JS bundle(s): %s", len(bundle_urls), bundle_urls)
        time.sleep(REQUEST_DELAY)
        for bundle_url in bundle_urls:
            if not robots.can_fetch(USER_AGENT, bundle_url):
                logger.warning("robots.txt disallows %s — skipping bundle", bundle_url)
                continue
            logger.info("Fetching JS bundle: %s", bundle_url)
            b_resp = session.get(bundle_url, timeout=REQUEST_TIMEOUT)
            logger.info("HTTP %s | %d bytes", b_resp.status_code, len(b_resp.content))
            if b_resp.status_code == 200:
                b_resp.encoding = "utf-8"
                bundle_texts.append(b_resp.text)
            else:
                site_failures.append({"source": SOURCE, "url": bundle_url, "status_code": b_resp.status_code,
                                      "error": "JS bundle not retrievable"})
            time.sleep(REQUEST_DELAY)
    except requests.RequestException as exc:
        logger.error("Request failed: %s", exc)
        site_failures.append({"source": SOURCE, "url": BASE_URL + "/",
                              "status_code": getattr(getattr(exc, "response", None), "status_code", None),
                              "error": str(exc)})

    # ---- 6.3 Parse author map and fact-check items from the bundle ----
    bundle_js = "\n".join(bundle_texts)
    author_map = {}
    for a_key, a_name, a_image, a_url in re.findall(
            r'([\w$]+):\{name:"([^"]+)",image:"([^"]*)",url:"([^"]*)"\}', bundle_js):
        author_map[a_key] = {"name": a_name, "url": a_url}
    logger.info("Authors found in bundle: %d", len(author_map))

    item_pattern = (r'\{id:(\d+),title:"((?:[^"\\]|\\.)*)",imageUrl:"([^"]*)",author:[\w$]+\.([\w$]+),'
                    r'category:"([^"]*)",verdict:"([^"]*)",date:"([^"]*)"\}')
    items = re.findall(item_pattern, bundle_js)
    site_stats["pages_discovered"] = 1 if bundle_texts else 0
    if not items:
        logger.error("Selector/pattern returned 0 fact-check items — bundle structure changed; re-inspect")
        site_stats["status"] = "FAILED_NO_ITEMS"
    logger.info("Found %d fact-check items in bundle", len(items))
    logger.info("Raw 'date' values in bundle: %s (stored as-is, not interpreted)",
                dict(Counter(item[6] for item in items)))

    seen_ids = set()
    for item_id, raw_title, image_url, author_key, category, verdict, raw_date in items:
        try:
            if item_id in seen_ids:
                site_stats["duplicate_urls"] += 1
                logger.warning("Duplicate item id skipped: %s", item_id)
                continue
            seen_ids.add(item_id)
            title = json.loads('"' + raw_title + '"')   # decode JS string escapes, keeps Devanagari
            dev_chars = len(re.findall(r"[\u0900-\u097F]", title))
            lat_chars = len(re.findall(r"[A-Za-z]", title))
            dev_ratio = dev_chars / max(dev_chars + lat_chars, 1)
            language = "ne" if dev_ratio >= 0.7 else ("en" if dev_ratio <= 0.3 else "mixed")
            author_info = author_map.get(author_key)
            if author_info is None:
                logger.warning("Author key '%s' not in author map (item %s)", author_key, item_id)
            record = {
                "source": SOURCE, "article_url": None, "article_id": f"tpfactcheck-{item_id}",
                "title": title, "claim": None, "verdict": verdict or None,
                "claimant": None, "claim_date": None, "fact_check_date": None,
                "published_date": raw_date or None,
                "author": author_info["name"] if author_info else None,
                "category": category or None, "language": language,
                "description": None, "article_text": None, "evidence": None, "sources": None,
                "correction": None, "image_url": image_url or None, "tags": None,
                "scraped_at": datetime.now(timezone.utc).isoformat(),
            }
            site_records.append(record)
            site_stats["successful"] += 1
        except Exception as exc:
            site_stats["failed"] += 1
            logger.exception("Parsing error for item %s: %s", item_id, exc)
            site_failures.append({"source": SOURCE, "url": f"{BASE_URL}/ (item {item_id})",
                                  "status_code": None, "error": repr(exc)})
    site_stats["urls_found"] = len(seen_ids) + site_stats["duplicate_urls"]
    logger.warning("Missing fields for all %s items: article_url, claim, article_text, evidence, sources "
                   "(not published in the site's data)", SOURCE)
    if site_stats["status"] == "STARTED":
        site_stats["status"] = "OK_LIMITED_FIELDS"

records_by_source[SOURCE] = site_records
failures_by_source[SOURCE] = site_failures
stats_by_source[SOURCE] = site_stats
logger.info("Successfully scraped %d items", site_stats["successful"])
logger.info("%s finished | %s", SOURCE, site_stats)

2026-11-22 15:19:59 | INFO | ============================================================
2026-11-22 15:19:59 | INFO | Starting TechPana Fact Check scraper
2026-11-22 15:19:59 | INFO | Fetching robots.txt: https://tpfactcheck.com/robots.txt
2026-11-22 15:20:00 | INFO | HTTP 404
2026-11-22 15:20:02 | INFO | Fetching page: https://tpfactcheck.com/
2026-11-22 15:20:03 | INFO | HTTP 200
2026-11-22 15:20:03 | WARNING | Dynamic rendering required — article list is not in the raw HTML. Reading the site's own static JS bundle instead (no browser needed).
2026-11-22 15:20:03 | INFO | Found 1 first-party JS bundle(s): ['https://tpfactcheck.com/assets/index-DtVYGj9T.js']
2026-11-22 15:20:04 | INFO | Fetching JS bundle: https://tpfactcheck.com/assets/index-DtVYGj9T.js
2026-11-22 15:20:06 | INFO | HTTP 200 | 285010 bytes
2026-11-22 15:20:08 | INFO | Authors found in bundle: 6
2026-11-22 15:20:08 | INFO | Found 115 fact-check items in bundle
2026-11-22 15:20:08 | INFO | Raw 'date' values in bundle: 

In [0]:
%python
# ============================================================
# STEP 7 — Scrape Karnali Factcheck (WordPress REST API)
# ============================================================
# Inspection findings:
#   * robots.txt only disallows /wp-admin/ -> /wp-json/ is allowed
#   * Same category scheme as Nepal Factcheck: "तथ्यजाँच" + verdict child categories
#   * Headings vary: "के छ दावी ?" / "के हो दावी?" (claim), "तथ्य के हो ?" (evidence), "तथ्यजाँच" (conclusion)
#   * Embedded WP author is often empty -> byline in a leading <ul><li><strong> list
#   * Media/guids are served from nepalfactcheck.org (shared network) -> excluded from sources
SOURCE = "Karnali Factcheck"
BASE_URL = "https://karnalifactcheck.org"
OWN_DOMAINS = {"karnalifactcheck.org", "www.karnalifactcheck.org"}
FACTCHECK_CATEGORY_NAME = "तथ्यजाँच"

site_records, site_failures = [], []
site_stats = {"source": SOURCE, "pages_discovered": 0, "urls_found": 0, "successful": 0,
              "skipped": 0, "failed": 0, "duplicate_urls": 0, "status": "STARTED"}
logger.info("=" * 60)
logger.info("Starting %s scraper", SOURCE)

# ---- 7.1 robots.txt ----
robots = urllib.robotparser.RobotFileParser()
robots_url = BASE_URL + "/robots.txt"
try:
    logger.info("Fetching robots.txt: %s", robots_url)
    resp = session.get(robots_url, timeout=REQUEST_TIMEOUT)
    logger.info("HTTP %s", resp.status_code)
    if resp.status_code in (401, 403):
        robots.disallow_all = True
    elif resp.status_code >= 400:
        robots.allow_all = True
    else:
        robots.parse(resp.text.splitlines())
except requests.RequestException as exc:
    logger.error("robots.txt request failed: %s -> treating site as disallowed", exc)
    robots.disallow_all = True
time.sleep(REQUEST_DELAY)

posts_api = BASE_URL + "/wp-json/wp/v2/posts"
if not robots.can_fetch(USER_AGENT, posts_api):
    logger.warning("robots.txt disallows %s — skipping %s", posts_api, SOURCE)
    site_stats["status"] = "BLOCKED_BY_ROBOTS"
else:
    # ---- 7.2 Category map ----
    categories = {}
    cat_url = BASE_URL + "/wp-json/wp/v2/categories"
    try:
        logger.info("Fetching categories: %s", cat_url)
        resp = session.get(cat_url, params={"per_page": 100}, timeout=REQUEST_TIMEOUT)
        logger.info("HTTP %s", resp.status_code)
        resp.raise_for_status()
        for cat in resp.json():
            categories[cat["id"]] = cat
    except (requests.RequestException, ValueError) as exc:
        logger.error("Could not load categories: %s", exc)
        site_failures.append({"source": SOURCE, "url": cat_url,
                              "status_code": getattr(getattr(exc, "response", None), "status_code", None),
                              "error": str(exc)})
    time.sleep(REQUEST_DELAY)

    factcheck_cat_ids = [cid for cid, cat in categories.items()
                         if html.unescape(cat["name"]).strip() == FACTCHECK_CATEGORY_NAME]
    verdict_cat_ids = {cid for cid, cat in categories.items() if cat.get("parent") in factcheck_cat_ids}
    logger.info("Fact-check category ids: %s | verdict categories: %s", factcheck_cat_ids,
                [categories[c]["name"] for c in verdict_cat_ids])
    if not factcheck_cat_ids:
        logger.error("Fact-check category '%s' not found — cannot filter fact checks", FACTCHECK_CATEGORY_NAME)
        site_stats["status"] = "FAILED_NO_CATEGORY"

    # ---- 7.3 Sitemap cross-check (informational only) ----
    try:
        sm_url = BASE_URL + "/sitemap_index.xml"
        logger.info("Fetching sitemap index for cross-check: %s", sm_url)
        resp = session.get(sm_url, timeout=REQUEST_TIMEOUT)
        logger.info("HTTP %s", resp.status_code)
        post_sitemaps = [u for u in re.findall(r"<loc>(.*?)</loc>", resp.text) if "post-sitemap" in u]
        sitemap_post_urls = set()
        for sm in post_sitemaps:
            time.sleep(REQUEST_DELAY)
            r_sm = session.get(sm, timeout=REQUEST_TIMEOUT)
            logger.info("Fetching sitemap: %s | HTTP %s", sm, r_sm.status_code)
            sitemap_post_urls.update(re.findall(r"<loc>(.*?)</loc>", r_sm.text))
        logger.info("Sitemap lists %d post URLs", len(sitemap_post_urls))
    except requests.RequestException as exc:
        logger.warning("Sitemap cross-check failed: %s", exc)
    time.sleep(REQUEST_DELAY)

    # ---- 7.4 Paginate the posts API ----
    raw_posts = []
    page = 1
    total_pages = None
    while factcheck_cat_ids and page <= MAX_LISTING_PAGES:
        params = {"per_page": 100, "page": page, "categories": ",".join(map(str, factcheck_cat_ids)),
                  "_embed": "author,wp:term,wp:featuredmedia", "orderby": "date", "order": "desc"}
        try:
            logger.info("Fetching page %d: %s", page, posts_api)
            resp = session.get(posts_api, params=params, timeout=REQUEST_TIMEOUT)
            logger.info("HTTP %s", resp.status_code)
            if resp.status_code == 400 and page > 1:
                logger.info("API reports no more pages (HTTP 400)")
                break
            resp.raise_for_status()
            page_posts = resp.json()
        except (requests.RequestException, ValueError) as exc:
            logger.error("Listing page %d failed: %s", page, exc)
            site_failures.append({"source": SOURCE, "url": f"{posts_api}?page={page}",
                                  "status_code": getattr(getattr(exc, "response", None), "status_code", None),
                                  "error": str(exc)})
            break
        if total_pages is None:
            total_pages = int(resp.headers.get("X-WP-TotalPages", 1))
            logger.info("API reports %s fact-check posts across %d pages",
                        resp.headers.get("X-WP-Total"), total_pages)
        if not page_posts:
            logger.info("Page %d returned no posts — stopping pagination", page)
            break
        raw_posts.extend(page_posts)
        site_stats["pages_discovered"] = page
        logger.info("Page %d: %d posts (running total %d)", page, len(page_posts), len(raw_posts))
        time.sleep(REQUEST_DELAY)
        if page >= total_pages or len(raw_posts) >= MAX_ARTICLES_PER_SITE:
            break
        page += 1

    # ---- 7.5 Deduplicate URLs ----
    seen_urls, unique_posts = set(), []
    for post in raw_posts:
        p_url = urlunparse(urlparse(post.get("link", ""))._replace(query="", fragment="")).rstrip("/").lower()
        if p_url in seen_urls:
            site_stats["duplicate_urls"] += 1
            logger.warning("Duplicate URL skipped: %s", post.get("link"))
            continue
        seen_urls.add(p_url)
        unique_posts.append(post)
    unique_posts = unique_posts[:MAX_ARTICLES_PER_SITE]
    site_stats["urls_found"] = len(unique_posts)
    logger.info("Found %d unique article URLs", len(unique_posts))

    # ---- 7.6 Parse each post ----
    for idx, post in enumerate(unique_posts):
        article_url = post.get("link")
        try:
            soup = BeautifulSoup(post["content"]["rendered"], "html.parser")
            for junk in soup.select("script, style, noscript, iframe, form, video, audio, .sharedaddy, "
                                    ".jp-relatedposts, .wp-video, .addtoany_share_save_container"):
                junk.decompose()

            author = None
            for emb_author in post.get("_embedded", {}).get("author", []):
                if emb_author.get("name"):
                    author = emb_author["name"].strip()
            first_el = soup.find(True)
            if first_el is not None and first_el.name == "ul" and first_el.find("strong") \
                    and len(first_el.get_text(strip=True)) < 60:
                if author is None:
                    author = first_el.get_text(" ", strip=True)
                first_el.decompose()

            source_links = []
            for a_tag in soup.find_all("a", href=True):
                href = a_tag["href"].strip()
                host = urlparse(href).netloc.lower()
                if href.startswith("http") and host and host not in OWN_DOMAINS \
                        and "nepalfactcheck.org" not in host and href not in source_links:
                    source_links.append(href)

            sections, all_blocks, seen_blocks = [], [], set()
            current_heading, current_texts = None, []
            for el in soup.find_all(["h2", "h3", "h4", "p", "li", "blockquote"]):
                txt = re.sub(r"\s+", " ", el.get_text(" ", strip=True)).strip()
                if not txt or txt in seen_blocks:
                    continue
                seen_blocks.add(txt)
                all_blocks.append(txt)
                bold_el = el.find(["strong", "b"]) if el.name == "p" else None
                bold_text = re.sub(r"\s+", " ", bold_el.get_text(" ", strip=True)).strip() if bold_el else ""
                is_pseudo_heading = (
                    bold_text != "" and len(bold_text) < 60 and txt.startswith(bold_text)
                    and re.search(r"दाबी|दावी|तथ्य", bold_text) is not None
                )
                if el.name in ("h2", "h3", "h4"):
                    sections.append((current_heading, current_texts))
                    current_heading, current_texts = txt, []
                elif is_pseudo_heading:
                    sections.append((current_heading, current_texts))
                    remainder = txt[len(bold_text):].strip()
                    current_heading, current_texts = bold_text, ([remainder] if remainder else [])
                else:
                    current_texts.append(txt)
            sections.append((current_heading, current_texts))

            claim = None
            for heading, texts in sections:
                if heading and re.search(r"दाबी|दावी", heading) and texts:
                    claim = "\n".join(texts)
                    break
            if claim is None:
                for heading, texts in sections:
                    if heading and "तथ्य" in heading:
                        break
                    if heading and texts:
                        claim = "\n".join(texts)
                        break

            evidence_parts, in_evidence = [], False
            for heading, texts in sections:
                if heading and "तथ्य" in heading:
                    in_evidence = True
                if in_evidence and texts:
                    evidence_parts.append(f"{heading}\n" + "\n".join(texts))
            evidence = "\n\n".join(evidence_parts) or None

            correction = None
            for heading, texts in sections:
                if heading and re.search(r"सच्याइ|संशोधन|correction", heading, re.I):
                    correction = "\n".join(texts) or heading

            verdict_labels, other_categories, tag_names = [], [], []
            for term_group in post.get("_embedded", {}).get("wp:term", []):
                for term in term_group:
                    term_name = html.unescape(term.get("name", "")).strip()
                    if term.get("taxonomy") == "category":
                        if term.get("id") in verdict_cat_ids:
                            verdict_labels.append(term_name)
                        elif term.get("id") not in factcheck_cat_ids:
                            other_categories.append(term_name)
                    elif term.get("taxonomy") == "post_tag":
                        tag_names.append(term_name)

            image_url = post.get("jetpack_featured_media_url") or None
            if not image_url:
                for media in post.get("_embedded", {}).get("wp:featuredmedia", []):
                    image_url = media.get("source_url") or image_url

            article_text = "\n\n".join(all_blocks)
            dev_chars = len(re.findall(r"[\u0900-\u097F]", article_text))
            lat_chars = len(re.findall(r"[A-Za-z]", article_text))
            dev_ratio = dev_chars / max(dev_chars + lat_chars, 1)
            language = "ne" if dev_ratio >= 0.7 else ("en" if dev_ratio <= 0.3 else "mixed")

            title = html.unescape(re.sub(r"<[^>]+>", "", post["title"]["rendered"])).strip()
            description = BeautifulSoup(post.get("excerpt", {}).get("rendered", ""), "html.parser").get_text(" ", strip=True) or None

            if idx < VALIDATE_HTML_SAMPLES and robots.can_fetch(USER_AGENT, article_url):
                logger.info("Validating metadata against HTML: %s", article_url)
                page_resp = session.get(article_url, timeout=REQUEST_TIMEOUT)
                logger.info("HTTP %s", page_resp.status_code)
                page_soup = BeautifulSoup(page_resp.text, "html.parser")
                visible_h1 = page_soup.find("h1")
                visible_title = visible_h1.get_text(" ", strip=True) if visible_h1 else None
                if visible_title and visible_title.strip() == title.strip():
                    logger.info("Visible <h1> matches API title")
                else:
                    logger.warning("Visible <h1> '%s' differs from API title '%s'", visible_title, title)
                time.sleep(REQUEST_DELAY)

            record = {
                "source": SOURCE, "article_url": article_url, "article_id": str(post.get("id")),
                "title": title, "claim": claim,
                "verdict": ", ".join(verdict_labels) or None,
                "claimant": None, "claim_date": None, "fact_check_date": None,
                "published_date": post.get("date"), "author": author,
                "category": ", ".join(other_categories) or None, "language": language,
                "description": description, "article_text": article_text or None,
                "evidence": evidence, "sources": " | ".join(source_links) or None,
                "correction": correction, "image_url": image_url,
                "tags": ", ".join(tag_names) or None,
                "scraped_at": datetime.now(timezone.utc).isoformat(),
            }
            missing = [f for f in ("title", "claim", "verdict", "article_text") if not record[f]]
            if missing:
                logger.warning("Missing fields %s | %s", missing, article_url)
            if not record["article_text"]:
                site_stats["skipped"] += 1
                logger.warning("Skipped (empty body): %s", article_url)
                continue
            site_records.append(record)
            site_stats["successful"] += 1
        except Exception as exc:
            site_stats["failed"] += 1
            logger.exception("Parsing error for %s: %s", article_url, exc)
            site_failures.append({"source": SOURCE, "url": article_url, "status_code": None, "error": repr(exc)})

    if site_stats["status"] == "STARTED":
        site_stats["status"] = "OK"

records_by_source[SOURCE] = site_records
failures_by_source[SOURCE] = site_failures
stats_by_source[SOURCE] = site_stats
logger.info("Successfully scraped %d articles", site_stats["successful"])
if site_stats["failed"]:
    logger.warning("%d articles failed parsing", site_stats["failed"])
logger.info("%s finished | %s", SOURCE, site_stats)

2026-11-22 15:20:18 | INFO | ============================================================
2026-11-22 15:20:18 | INFO | Starting Karnali Factcheck scraper
2026-11-22 15:20:18 | INFO | Fetching robots.txt: https://karnalifactcheck.org/robots.txt
2026-11-22 15:20:18 | INFO | HTTP 200
2026-11-22 15:20:20 | INFO | Fetching categories: https://karnalifactcheck.org/wp-json/wp/v2/categories
2026-11-22 15:20:20 | INFO | HTTP 200
2026-11-22 15:20:22 | INFO | Fact-check category ids: [1] | verdict categories: ['सही सूचना', 'मिथ्या सूचना', 'अपुष्ट सूचना', 'भ्रामक सूचना']
2026-11-22 15:20:22 | INFO | Fetching sitemap index for cross-check: https://karnalifactcheck.org/sitemap_index.xml
2026-11-22 15:20:22 | INFO | HTTP 200
2026-11-22 15:20:24 | INFO | Fetching sitemap: https://karnalifactcheck.org/post-sitemap1.xml | HTTP 200
2026-11-22 15:20:26 | INFO | Fetching sitemap: https://karnalifactcheck.org/post-sitemap2.xml | HTTP 200
2026-11-22 15:20:26 | INFO | Sitemap lists 361 post URLs
2026-11-22 15

In [0]:
%python
# ============================================================
# STEP 8 — Scrape Lumbini Factcheck (WordPress REST API)
# ============================================================
# Inspection findings:
#   * robots.txt only disallows /wp-admin/ -> /wp-json/ is allowed; small site (~30 posts)
#   * Same category scheme ("तथ्यजाँच" + verdict child categories)
#   * Claim headings are NOT standardised: "के छ भाइरल भिडियोमा ?", "कसरी भयो भाइरल?",
#     "कहाँ के लेखियो?" -> fall back to the first headed section before the first "तथ्य" heading
#   * Byline is a leading <ul><li><strong>Name</strong></li></ul>
SOURCE = "Lumbini Factcheck"
BASE_URL = "https://lumbinifactcheck.org"
OWN_DOMAINS = {"lumbinifactcheck.org", "www.lumbinifactcheck.org"}
FACTCHECK_CATEGORY_NAME = "तथ्यजाँच"

site_records, site_failures = [], []
site_stats = {"source": SOURCE, "pages_discovered": 0, "urls_found": 0, "successful": 0,
              "skipped": 0, "failed": 0, "duplicate_urls": 0, "status": "STARTED"}
logger.info("=" * 60)
logger.info("Starting %s scraper", SOURCE)

# ---- 8.1 robots.txt ----
robots = urllib.robotparser.RobotFileParser()
robots_url = BASE_URL + "/robots.txt"
try:
    logger.info("Fetching robots.txt: %s", robots_url)
    resp = session.get(robots_url, timeout=REQUEST_TIMEOUT)
    logger.info("HTTP %s", resp.status_code)
    if resp.status_code in (401, 403):
        robots.disallow_all = True
    elif resp.status_code >= 400:
        robots.allow_all = True
    else:
        robots.parse(resp.text.splitlines())
except requests.RequestException as exc:
    logger.error("robots.txt request failed: %s -> treating site as disallowed", exc)
    robots.disallow_all = True
time.sleep(REQUEST_DELAY)

posts_api = BASE_URL + "/wp-json/wp/v2/posts"
if not robots.can_fetch(USER_AGENT, posts_api):
    logger.warning("robots.txt disallows %s — skipping %s", posts_api, SOURCE)
    site_stats["status"] = "BLOCKED_BY_ROBOTS"
else:
    # ---- 8.2 Category map ----
    categories = {}
    cat_url = BASE_URL + "/wp-json/wp/v2/categories"
    try:
        logger.info("Fetching categories: %s", cat_url)
        resp = session.get(cat_url, params={"per_page": 100}, timeout=REQUEST_TIMEOUT)
        logger.info("HTTP %s", resp.status_code)
        resp.raise_for_status()
        for cat in resp.json():
            categories[cat["id"]] = cat
    except (requests.RequestException, ValueError) as exc:
        logger.error("Could not load categories: %s", exc)
        site_failures.append({"source": SOURCE, "url": cat_url,
                              "status_code": getattr(getattr(exc, "response", None), "status_code", None),
                              "error": str(exc)})
    time.sleep(REQUEST_DELAY)

    factcheck_cat_ids = [cid for cid, cat in categories.items()
                         if html.unescape(cat["name"]).strip() == FACTCHECK_CATEGORY_NAME]
    verdict_cat_ids = {cid for cid, cat in categories.items() if cat.get("parent") in factcheck_cat_ids}
    logger.info("Fact-check category ids: %s | verdict categories: %s", factcheck_cat_ids,
                [categories[c]["name"] for c in verdict_cat_ids])
    if not factcheck_cat_ids:
        logger.error("Fact-check category '%s' not found — cannot filter fact checks", FACTCHECK_CATEGORY_NAME)
        site_stats["status"] = "FAILED_NO_CATEGORY"

    # ---- 8.3 Sitemap cross-check (informational only) ----
    try:
        sm_url = BASE_URL + "/sitemap_index.xml"
        logger.info("Fetching sitemap index for cross-check: %s", sm_url)
        resp = session.get(sm_url, timeout=REQUEST_TIMEOUT)
        logger.info("HTTP %s", resp.status_code)
        post_sitemaps = [u for u in re.findall(r"<loc>(.*?)</loc>", resp.text) if "post-sitemap" in u]
        sitemap_post_urls = set()
        for sm in post_sitemaps:
            time.sleep(REQUEST_DELAY)
            r_sm = session.get(sm, timeout=REQUEST_TIMEOUT)
            logger.info("Fetching sitemap: %s | HTTP %s", sm, r_sm.status_code)
            sitemap_post_urls.update(re.findall(r"<loc>(.*?)</loc>", r_sm.text))
        logger.info("Sitemap lists %d post URLs", len(sitemap_post_urls))
    except requests.RequestException as exc:
        logger.warning("Sitemap cross-check failed: %s", exc)
    time.sleep(REQUEST_DELAY)

    # ---- 8.4 Paginate the posts API ----
    raw_posts = []
    page = 1
    total_pages = None
    while factcheck_cat_ids and page <= MAX_LISTING_PAGES:
        params = {"per_page": 100, "page": page, "categories": ",".join(map(str, factcheck_cat_ids)),
                  "_embed": "author,wp:term,wp:featuredmedia", "orderby": "date", "order": "desc"}
        try:
            logger.info("Fetching page %d: %s", page, posts_api)
            resp = session.get(posts_api, params=params, timeout=REQUEST_TIMEOUT)
            logger.info("HTTP %s", resp.status_code)
            if resp.status_code == 400 and page > 1:
                logger.info("API reports no more pages (HTTP 400)")
                break
            resp.raise_for_status()
            page_posts = resp.json()
        except (requests.RequestException, ValueError) as exc:
            logger.error("Listing page %d failed: %s", page, exc)
            site_failures.append({"source": SOURCE, "url": f"{posts_api}?page={page}",
                                  "status_code": getattr(getattr(exc, "response", None), "status_code", None),
                                  "error": str(exc)})
            break
        if total_pages is None:
            total_pages = int(resp.headers.get("X-WP-TotalPages", 1))
            logger.info("API reports %s fact-check posts across %d pages",
                        resp.headers.get("X-WP-Total"), total_pages)
        if not page_posts:
            logger.info("Page %d returned no posts — stopping pagination", page)
            break
        raw_posts.extend(page_posts)
        site_stats["pages_discovered"] = page
        logger.info("Page %d: %d posts (running total %d)", page, len(page_posts), len(raw_posts))
        time.sleep(REQUEST_DELAY)
        if page >= total_pages or len(raw_posts) >= MAX_ARTICLES_PER_SITE:
            break
        page += 1

    # ---- 8.5 Deduplicate URLs ----
    seen_urls, unique_posts = set(), []
    for post in raw_posts:
        p_url = urlunparse(urlparse(post.get("link", ""))._replace(query="", fragment="")).rstrip("/").lower()
        if p_url in seen_urls:
            site_stats["duplicate_urls"] += 1
            logger.warning("Duplicate URL skipped: %s", post.get("link"))
            continue
        seen_urls.add(p_url)
        unique_posts.append(post)
    unique_posts = unique_posts[:MAX_ARTICLES_PER_SITE]
    site_stats["urls_found"] = len(unique_posts)
    logger.info("Found %d unique article URLs", len(unique_posts))

    # ---- 8.6 Parse each post ----
    for idx, post in enumerate(unique_posts):
        article_url = post.get("link")
        try:
            soup = BeautifulSoup(post["content"]["rendered"], "html.parser")
            for junk in soup.select("script, style, noscript, iframe, form, video, audio, .sharedaddy, "
                                    ".jp-relatedposts, .wp-video, .addtoany_share_save_container"):
                junk.decompose()

            author = None
            for emb_author in post.get("_embedded", {}).get("author", []):
                if emb_author.get("name"):
                    author = emb_author["name"].strip()
            first_el = soup.find(True)
            if first_el is not None and first_el.name == "ul" and first_el.find("strong") \
                    and len(first_el.get_text(strip=True)) < 60:
                if author is None:
                    author = first_el.get_text(" ", strip=True)
                first_el.decompose()

            source_links = []
            for a_tag in soup.find_all("a", href=True):
                href = a_tag["href"].strip()
                host = urlparse(href).netloc.lower()
                if href.startswith("http") and host and host not in OWN_DOMAINS \
                        and "nepalfactcheck.org" not in host and href not in source_links:
                    source_links.append(href)

            sections, all_blocks, seen_blocks = [], [], set()
            current_heading, current_texts = None, []
            for el in soup.find_all(["h2", "h3", "h4", "p", "li", "blockquote"]):
                txt = re.sub(r"\s+", " ", el.get_text(" ", strip=True)).strip()
                if not txt or txt in seen_blocks:
                    continue
                seen_blocks.add(txt)
                all_blocks.append(txt)
                bold_el = el.find(["strong", "b"]) if el.name == "p" else None
                bold_text = re.sub(r"\s+", " ", bold_el.get_text(" ", strip=True)).strip() if bold_el else ""
                is_pseudo_heading = (
                    bold_text != "" and len(bold_text) < 60 and txt.startswith(bold_text)
                    and re.search(r"दाबी|दावी|तथ्य", bold_text) is not None
                )
                if el.name in ("h2", "h3", "h4"):
                    sections.append((current_heading, current_texts))
                    current_heading, current_texts = txt, []
                elif is_pseudo_heading:
                    sections.append((current_heading, current_texts))
                    remainder = txt[len(bold_text):].strip()
                    current_heading, current_texts = bold_text, ([remainder] if remainder else [])
                else:
                    current_texts.append(txt)
            sections.append((current_heading, current_texts))

            claim = None
            for heading, texts in sections:
                if heading and re.search(r"दाबी|दावी", heading) and texts:
                    claim = "\n".join(texts)
                    break
            if claim is None:
                for heading, texts in sections:
                    if heading and "तथ्य" in heading:
                        break
                    if heading and texts:
                        claim = "\n".join(texts)
                        break

            evidence_parts, in_evidence = [], False
            for heading, texts in sections:
                if heading and "तथ्य" in heading:
                    in_evidence = True
                if in_evidence and texts:
                    evidence_parts.append(f"{heading}\n" + "\n".join(texts))
            evidence = "\n\n".join(evidence_parts) or None

            correction = None
            for heading, texts in sections:
                if heading and re.search(r"सच्याइ|संशोधन|correction", heading, re.I):
                    correction = "\n".join(texts) or heading

            verdict_labels, other_categories, tag_names = [], [], []
            for term_group in post.get("_embedded", {}).get("wp:term", []):
                for term in term_group:
                    term_name = html.unescape(term.get("name", "")).strip()
                    if term.get("taxonomy") == "category":
                        if term.get("id") in verdict_cat_ids:
                            verdict_labels.append(term_name)
                        elif term.get("id") not in factcheck_cat_ids:
                            other_categories.append(term_name)
                    elif term.get("taxonomy") == "post_tag":
                        tag_names.append(term_name)

            image_url = post.get("jetpack_featured_media_url") or None
            if not image_url:
                for media in post.get("_embedded", {}).get("wp:featuredmedia", []):
                    image_url = media.get("source_url") or image_url

            article_text = "\n\n".join(all_blocks)
            dev_chars = len(re.findall(r"[\u0900-\u097F]", article_text))
            lat_chars = len(re.findall(r"[A-Za-z]", article_text))
            dev_ratio = dev_chars / max(dev_chars + lat_chars, 1)
            language = "ne" if dev_ratio >= 0.7 else ("en" if dev_ratio <= 0.3 else "mixed")

            title = html.unescape(re.sub(r"<[^>]+>", "", post["title"]["rendered"])).strip()
            description = BeautifulSoup(post.get("excerpt", {}).get("rendered", ""), "html.parser").get_text(" ", strip=True) or None

            if idx < VALIDATE_HTML_SAMPLES and robots.can_fetch(USER_AGENT, article_url):
                logger.info("Validating metadata against HTML: %s", article_url)
                page_resp = session.get(article_url, timeout=REQUEST_TIMEOUT)
                logger.info("HTTP %s", page_resp.status_code)
                page_soup = BeautifulSoup(page_resp.text, "html.parser")
                visible_h1 = page_soup.find("h1")
                visible_title = visible_h1.get_text(" ", strip=True) if visible_h1 else None
                if visible_title and visible_title.strip() == title.strip():
                    logger.info("Visible <h1> matches API title")
                else:
                    logger.warning("Visible <h1> '%s' differs from API title '%s'", visible_title, title)
                time.sleep(REQUEST_DELAY)

            record = {
                "source": SOURCE, "article_url": article_url, "article_id": str(post.get("id")),
                "title": title, "claim": claim,
                "verdict": ", ".join(verdict_labels) or None,
                "claimant": None, "claim_date": None, "fact_check_date": None,
                "published_date": post.get("date"), "author": author,
                "category": ", ".join(other_categories) or None, "language": language,
                "description": description, "article_text": article_text or None,
                "evidence": evidence, "sources": " | ".join(source_links) or None,
                "correction": correction, "image_url": image_url,
                "tags": ", ".join(tag_names) or None,
                "scraped_at": datetime.now(timezone.utc).isoformat(),
            }
            missing = [f for f in ("title", "claim", "verdict", "article_text") if not record[f]]
            if missing:
                logger.warning("Missing fields %s | %s", missing, article_url)
            if not record["article_text"]:
                site_stats["skipped"] += 1
                logger.warning("Skipped (empty body): %s", article_url)
                continue
            site_records.append(record)
            site_stats["successful"] += 1
        except Exception as exc:
            site_stats["failed"] += 1
            logger.exception("Parsing error for %s: %s", article_url, exc)
            site_failures.append({"source": SOURCE, "url": article_url, "status_code": None, "error": repr(exc)})

    if site_stats["status"] == "STARTED":
        site_stats["status"] = "OK"

records_by_source[SOURCE] = site_records
failures_by_source[SOURCE] = site_failures
stats_by_source[SOURCE] = site_stats
logger.info("Successfully scraped %d articles", site_stats["successful"])
if site_stats["failed"]:
    logger.warning("%d articles failed parsing", site_stats["failed"])
logger.info("%s finished | %s", SOURCE, site_stats)

2026-11-22 15:21:08 | INFO | ============================================================
2026-11-22 15:21:08 | INFO | Starting Lumbini Factcheck scraper
2026-11-22 15:21:08 | INFO | Fetching robots.txt: https://lumbinifactcheck.org/robots.txt
2026-11-22 15:21:08 | INFO | HTTP 200
2026-11-22 15:21:10 | INFO | Fetching categories: https://lumbinifactcheck.org/wp-json/wp/v2/categories
2026-11-22 15:21:13 | INFO | HTTP 200
2026-11-22 15:21:14 | INFO | Fact-check category ids: [1] | verdict categories: ['सही सूचना', 'मिथ्या सूचना', 'अपुष्ट सूचना', 'भ्रामक सूचना']
2026-11-22 15:21:14 | INFO | Fetching sitemap index for cross-check: https://lumbinifactcheck.org/sitemap_index.xml
2026-11-22 15:21:14 | INFO | HTTP 200
2026-11-22 15:21:16 | INFO | Fetching sitemap: https://lumbinifactcheck.org/post-sitemap.xml | HTTP 200
2026-11-22 15:21:16 | INFO | Sitemap lists 30 post URLs
2026-11-22 15:21:18 | INFO | Fetching page 1: https://lumbinifactcheck.org/wp-json/wp/v2/posts
2026-11-22 15:21:20 | INF

In [0]:
%python
# ============================================================
# STEP 9 — Scrape Sachilo (sitemap + schema.org ClaimReview)
# ============================================================
# Inspection findings:
#   * robots.txt allows / but disallows /admin, /api/, /check/, /sign-in
#   * sitemap.xml lists verdict pages under /v/<slug>; other entries (about, privacy...) excluded
#   * Each /v/ page embeds JSON-LD: NewsArticle (headline, description, dates) and one
#     ClaimReview per claim (claimReviewed, reviewRating.alternateName, ratingExplanation,
#     itemReviewed.url / author / datePublished). One page can hold several claims.
#   * Visible verdict label is in <header class="verdict-banner"> after "Published / Checked"
#   * Visible body (claims + evidence cards) follows the <h1>; nav/footer text precedes it
SOURCE = "Sachilo"
BASE_URL = "https://www.sachilo.com"

site_records, site_failures = [], []
site_stats = {"source": SOURCE, "pages_discovered": 0, "urls_found": 0, "successful": 0,
              "skipped": 0, "failed": 0, "duplicate_urls": 0, "status": "STARTED"}
logger.info("=" * 60)
logger.info("Starting %s scraper", SOURCE)

# ---- 9.1 robots.txt ----
robots = urllib.robotparser.RobotFileParser()
robots_url = BASE_URL + "/robots.txt"
try:
    logger.info("Fetching robots.txt: %s", robots_url)
    resp = session.get(robots_url, timeout=REQUEST_TIMEOUT)
    logger.info("HTTP %s", resp.status_code)
    if resp.status_code in (401, 403):
        robots.disallow_all = True
    elif resp.status_code >= 400:
        robots.allow_all = True
    else:
        robots.parse(resp.text.splitlines())
except requests.RequestException as exc:
    logger.error("robots.txt request failed: %s -> treating site as disallowed", exc)
    robots.disallow_all = True
time.sleep(REQUEST_DELAY)

# ---- 9.2 Discover article URLs from sitemap ----
article_urls = []
sitemap_url = BASE_URL + "/sitemap.xml"
if not robots.can_fetch(USER_AGENT, sitemap_url):
    logger.warning("robots.txt disallows %s — skipping %s", sitemap_url, SOURCE)
    site_stats["status"] = "BLOCKED_BY_ROBOTS"
else:
    try:
        logger.info("Fetching sitemap: %s", sitemap_url)
        resp = session.get(sitemap_url, timeout=REQUEST_TIMEOUT)
        logger.info("HTTP %s", resp.status_code)
        resp.raise_for_status()
        all_locs = re.findall(r"<loc>(.*?)</loc>", resp.text)
        site_stats["pages_discovered"] = 1
        logger.info("Sitemap entries: %d", len(all_locs))
        seen_urls = set()
        for loc in all_locs:
            loc = html.unescape(loc.strip())
            if "/v/" not in urlparse(loc).path:
                continue  # not a verdict page
            norm = urlunparse(urlparse(loc)._replace(query="", fragment="")).rstrip("/").lower()
            if norm in seen_urls:
                site_stats["duplicate_urls"] += 1
                logger.warning("Duplicate URL skipped: %s", loc)
                continue
            if not robots.can_fetch(USER_AGENT, loc):
                site_stats["skipped"] += 1
                logger.warning("robots.txt disallows %s — skipped", loc)
                continue
            seen_urls.add(norm)
            article_urls.append(loc)
    except requests.RequestException as exc:
        logger.error("Sitemap request failed: %s", exc)
        site_failures.append({"source": SOURCE, "url": sitemap_url,
                              "status_code": getattr(getattr(exc, "response", None), "status_code", None),
                              "error": str(exc)})
    time.sleep(REQUEST_DELAY)

article_urls = article_urls[:MAX_ARTICLES_PER_SITE]
site_stats["urls_found"] = len(article_urls)
logger.info("Found %d article URLs", len(article_urls))

# ---- 9.3 Fetch and parse each verdict page ----
for idx, article_url in enumerate(article_urls):
    status_code = None
    try:
        logger.info("[%d/%d] Fetching page: %s", idx + 1, len(article_urls), article_url)
        resp = session.get(article_url, timeout=REQUEST_TIMEOUT)
        status_code = resp.status_code
        logger.info("HTTP %s", resp.status_code)
        time.sleep(REQUEST_DELAY)
        if resp.status_code != 200:
            site_stats["failed"] += 1
            site_failures.append({"source": SOURCE, "url": article_url, "status_code": resp.status_code,
                                  "error": "Non-200 response"})
            continue
        resp.encoding = "utf-8"
        soup = BeautifulSoup(resp.text, "html.parser")

        # JSON-LD blocks
        ld_items = []
        for ld_tag in soup.find_all("script", type="application/ld+json"):
            try:
                ld_data = json.loads(ld_tag.string or ld_tag.get_text())
            except ValueError as exc:
                logger.warning("Invalid JSON-LD on %s: %s", article_url, exc)
                continue
            if isinstance(ld_data, list):
                ld_items.extend(ld_data)
            elif isinstance(ld_data, dict) and "@graph" in ld_data:
                ld_items.extend(ld_data["@graph"])
            else:
                ld_items.append(ld_data)
        news_ld = {}
        for ld in ld_items:
            if isinstance(ld, dict) and ld.get("@type") in ("NewsArticle", "Article") and not news_ld:
                news_ld = ld
        claim_reviews = [ld for ld in ld_items if isinstance(ld, dict) and ld.get("@type") == "ClaimReview"]
        if not claim_reviews:
            logger.warning("No ClaimReview JSON-LD found on %s", article_url)

        meta = {}
        for m_tag in soup.find_all("meta"):
            m_key = m_tag.get("property") or m_tag.get("name")
            if m_key and m_tag.get("content"):
                meta[m_key] = m_tag["content"]

        # Visible title, validated against JSON-LD headline
        h1_tag = soup.find("h1")
        visible_title = h1_tag.get_text(" ", strip=True) if h1_tag else None
        headline = (news_ld.get("headline") or "").strip() or None
        if visible_title is None:
            logger.warning("Selector h1 returned 0 results on %s", article_url)
        elif headline and visible_title != headline:
            logger.info("Visible <h1> differs from JSON-LD headline (using <h1>) | %s", article_url)
        title = visible_title or headline

        # Visible verdict label from the banner (fallback: prefix of twitter:title, e.g. "Unverifiable: ...")
        verdict = None
        banner = soup.select_one("header.verdict-banner")
        if banner is None:
            logger.warning("Selector header.verdict-banner returned 0 results on %s", article_url)
        else:
            banner_lines = [ln for ln in banner.get_text("\n", strip=True).split("\n") if ln.strip()]
            if "Checked" in banner_lines:
                checked_pos = banner_lines.index("Checked")
                if checked_pos + 2 < len(banner_lines):
                    verdict = banner_lines[checked_pos + 2]
        meta_verdict = meta.get("twitter:title", "").split(":")[0].strip() if ":" in meta.get("twitter:title", "") else None
        if verdict is None:
            verdict = meta_verdict

        # Claims (one per ClaimReview) and their per-claim verdict + explanation
        claim_lines, evidence_lines, claimants, claim_dates, reviewed_urls = [], [], [], [], []
        for n, review in enumerate(claim_reviews, start=1):
            rating = review.get("reviewRating") or {}
            item = review.get("itemReviewed") or {}
            if review.get("claimReviewed"):
                claim_lines.append(f"[{n}] {review['claimReviewed']}")
            evidence_lines.append(f"[{n}] {rating.get('alternateName')}: {rating.get('ratingExplanation')}")
            item_author = (item.get("author") or {}).get("name") if isinstance(item.get("author"), dict) else None
            if item_author and item_author not in claimants:
                claimants.append(item_author)
            if item.get("datePublished") and item["datePublished"] not in claim_dates:
                claim_dates.append(item["datePublished"])
            if item.get("url") and item["url"] not in reviewed_urls:
                reviewed_urls.append(item["url"])

        # Visible article text: everything from the <h1> onward (nav + footer come before it)
        for junk in soup(["script", "style", "noscript", "svg"]):
            junk.decompose()
        body_lines = [ln.strip() for ln in (soup.body.get_text("\n", strip=True) if soup.body else "").split("\n")]
        start_pos = body_lines.index(visible_title) if visible_title in body_lines else 0
        if visible_title not in body_lines:
            logger.warning("Could not locate <h1> in body text; using full body | %s", article_url)
        clean_lines = []
        for ln in body_lines[start_pos:]:
            if not ln or re.fullmatch(r"[\W_]+", ln):
                continue
            if clean_lines and clean_lines[-1] == ln:
                continue
            clean_lines.append(ln)
        article_text = "\n".join(clean_lines) or None

        # External sources cited in the evidence cards + the reviewed article(s)
        source_links = list(reviewed_urls)
        for a_tag in soup.find_all("a", href=True):
            href = a_tag["href"].strip()
            host = urlparse(href).netloc.lower()
            if href.startswith("http") and host and "sachilo.com" not in host and href not in source_links:
                source_links.append(href)

        dev_chars = len(re.findall(r"[\u0900-\u097F]", article_text or ""))
        lat_chars = len(re.findall(r"[A-Za-z]", article_text or ""))
        dev_ratio = dev_chars / max(dev_chars + lat_chars, 1)
        language = "ne" if dev_ratio >= 0.7 else ("en" if dev_ratio <= 0.3 else "mixed")

        news_author = news_ld.get("author")
        record = {
            "source": SOURCE, "article_url": article_url,
            "article_id": urlparse(article_url).path.rstrip("/").split("-")[-1],
            "title": title, "claim": "\n".join(claim_lines) or None, "verdict": verdict,
            "claimant": ", ".join(claimants) or None,
            "claim_date": claim_dates[0] if claim_dates else None,
            "fact_check_date": claim_reviews[0].get("datePublished") if claim_reviews else None,
            "published_date": news_ld.get("datePublished") or meta.get("article:published_time"),
            "author": news_author.get("name") if isinstance(news_author, dict) else meta.get("author"),
            "category": meta.get("article:section"), "language": language,
            "description": news_ld.get("description") or meta.get("og:description"),
            "article_text": article_text, "evidence": "\n".join(evidence_lines) or None,
            "sources": " | ".join(source_links) or None, "correction": None,
            "image_url": news_ld.get("image") if isinstance(news_ld.get("image"), str) else meta.get("og:image"),
            "tags": meta.get("keywords"),
            "scraped_at": datetime.now(timezone.utc).isoformat(),
        }
        missing = [f for f in ("title", "claim", "verdict", "article_text") if not record[f]]
        if missing:
            logger.warning("Missing fields %s | %s", missing, article_url)
        if not record["article_text"] and not record["claim"]:
            site_stats["skipped"] += 1
            logger.warning("Skipped (no claim and no body): %s", article_url)
            continue
        site_records.append(record)
        site_stats["successful"] += 1
    except requests.RequestException as exc:
        site_stats["failed"] += 1
        logger.error("HTTP failure for %s: %s", article_url, exc)
        site_failures.append({"source": SOURCE, "url": article_url,
                              "status_code": getattr(getattr(exc, "response", None), "status_code", status_code),
                              "error": str(exc)})
    except Exception as exc:
        site_stats["failed"] += 1
        logger.exception("Parsing error for %s: %s", article_url, exc)
        site_failures.append({"source": SOURCE, "url": article_url, "status_code": status_code, "error": repr(exc)})

if site_stats["status"] == "STARTED":
    site_stats["status"] = "OK"
records_by_source[SOURCE] = site_records
failures_by_source[SOURCE] = site_failures
stats_by_source[SOURCE] = site_stats
logger.info("Successfully scraped %d articles", site_stats["successful"])
if site_stats["failed"]:
    logger.warning("%d articles failed", site_stats["failed"])
logger.info("%s finished | %s", SOURCE, site_stats)

2026-11-22 15:21:34 | INFO | ============================================================
2026-11-22 15:21:34 | INFO | Starting Sachilo scraper
2026-11-22 15:21:34 | INFO | Fetching robots.txt: https://www.sachilo.com/robots.txt
2026-11-22 15:21:36 | INFO | HTTP 200
2026-11-22 15:21:37 | INFO | Fetching sitemap: https://www.sachilo.com/sitemap.xml
2026-11-22 15:21:38 | INFO | HTTP 200
2026-11-22 15:21:38 | INFO | Sitemap entries: 89
2026-11-22 15:21:39 | INFO | Found 84 article URLs
2026-11-22 15:21:39 | INFO | [1/84] Fetching page: https://www.sachilo.com/v/php-vulnerabilities-and-their-global-impact-on-cybersecurity-cmvmn0yvp0000jwxy1p6ltyun
2026-11-22 15:21:44 | INFO | HTTP 200
2026-11-22 15:21:46 | INFO | [2/84] Fetching page: https://www.sachilo.com/v/collision-of-two-sf-260-training-aircraft-in-philippines-cmvmibi16000008l7zped61f3
2026-11-22 15:21:48 | INFO | HTTP 200
2026-11-22 15:21:50 | INFO | [3/84] Fetching page: https://www.sachilo.com/v/claims-about-nepal-s-leadership-and

In [0]:
%python
# ============================================================
# STEP 10 — Scrape NepalCheck
# ============================================================
# Inspection findings: WordPress.com site whose robots.txt is "User-agent: * / Disallow: /".
# We respect robots.txt, so no content is requested beyond robots.txt itself.
SOURCE = "NepalCheck"
BASE_URL = "https://nepalcheck.org"

site_records, site_failures = [], []
site_stats = {"source": SOURCE, "pages_discovered": 0, "urls_found": 0, "successful": 0,
              "skipped": 0, "failed": 0, "duplicate_urls": 0, "status": "STARTED"}
logger.info("=" * 60)
logger.info("Starting %s scraper", SOURCE)

robots = urllib.robotparser.RobotFileParser()
robots_url = BASE_URL + "/robots.txt"
try:
    logger.info("Fetching robots.txt: %s", robots_url)
    resp = session.get(robots_url, timeout=REQUEST_TIMEOUT)
    logger.info("HTTP %s", resp.status_code)
    if resp.status_code in (401, 403):
        robots.disallow_all = True
    elif resp.status_code >= 400:
        robots.allow_all = True
    else:
        robots.parse(resp.text.splitlines())
except requests.RequestException as exc:
    logger.error("robots.txt request failed: %s -> treating site as disallowed", exc)
    robots.disallow_all = True
time.sleep(REQUEST_DELAY)

if not robots.can_fetch(USER_AGENT, BASE_URL + "/"):
    logger.warning("robots.txt disallows all crawling of %s — respecting it, skipping %s", BASE_URL, SOURCE)
    site_stats["status"] = "BLOCKED_BY_ROBOTS"
    site_failures.append({"source": SOURCE, "url": robots_url, "status_code": resp.status_code if "resp" in globals() else None,
                          "error": "robots.txt Disallow: / — not scraped"})
else:
    logger.warning("robots.txt now allows crawling — inspect %s and add site-specific selectors here", SOURCE)
    site_stats["status"] = "ALLOWED_NOT_IMPLEMENTED"

records_by_source[SOURCE] = site_records
failures_by_source[SOURCE] = site_failures
stats_by_source[SOURCE] = site_stats
logger.info("%s finished | %s", SOURCE, site_stats)

2026-11-22 15:26:52 | INFO | ============================================================
2026-11-22 15:26:52 | INFO | Starting NepalCheck scraper
2026-11-22 15:26:52 | INFO | Fetching robots.txt: https://nepalcheck.org/robots.txt
2026-11-22 15:26:53 | INFO | HTTP 200
2026-11-22 15:26:54 | WARNING | robots.txt disallows all crawling of https://nepalcheck.org — respecting it, skipping NepalCheck
2026-11-22 15:26:54 | INFO | NepalCheck finished | {'source': 'NepalCheck', 'pages_discovered': 0, 'urls_found': 0, 'successful': 0, 'skipped': 0, 'failed': 0, 'duplicate_urls': 0, 'status': 'BLOCKED_BY_ROBOTS'}


In [0]:
%python
# ============================================================
# STEP 11 — Scrape South Asia Check
# ============================================================
# Inspection findings: robots.txt has only content-signal comments (no Disallow rules),
# but the homepage now serves unrelated (Thai casino / link-farm) content: no Nepali text,
# no fact-check articles, ~all links point to external spam domains. The domain appears
# to no longer host South Asia Check. We verify this at run time and skip if still true.
SOURCE = "South Asia Check"
BASE_URL = "https://southasiacheck.org"

site_records, site_failures = [], []
site_stats = {"source": SOURCE, "pages_discovered": 0, "urls_found": 0, "successful": 0,
              "skipped": 0, "failed": 0, "duplicate_urls": 0, "status": "STARTED"}
logger.info("=" * 60)
logger.info("Starting %s scraper", SOURCE)

robots = urllib.robotparser.RobotFileParser()
robots_url = BASE_URL + "/robots.txt"
try:
    logger.info("Fetching robots.txt: %s", robots_url)
    resp = session.get(robots_url, timeout=REQUEST_TIMEOUT)
    logger.info("HTTP %s", resp.status_code)
    if resp.status_code in (401, 403):
        robots.disallow_all = True
    elif resp.status_code >= 400:
        robots.allow_all = True
    else:
        robots.parse(resp.text.splitlines())
except requests.RequestException as exc:
    logger.error("robots.txt request failed: %s -> treating site as disallowed", exc)
    robots.disallow_all = True
time.sleep(REQUEST_DELAY)

if not robots.can_fetch(USER_AGENT, BASE_URL + "/"):
    logger.warning("robots.txt disallows %s — skipping", BASE_URL)
    site_stats["status"] = "BLOCKED_BY_ROBOTS"
else:
    try:
        logger.info("Fetching page: %s", BASE_URL + "/")
        resp = session.get(BASE_URL + "/", timeout=REQUEST_TIMEOUT)
        logger.info("HTTP %s", resp.status_code)
        resp.encoding = resp.apparent_encoding
        home_soup = BeautifulSoup(resp.text, "html.parser")
        page_title = home_soup.title.get_text(strip=True) if home_soup.title else ""
        hrefs = [a["href"] for a in home_soup.find_all("a", href=True)]
        internal_links = [h for h in hrefs if h.startswith("/") or "southasiacheck.org" in urlparse(h).netloc]
        page_text = home_soup.get_text(" ", strip=True)
        devanagari_chars = len(re.findall(r"[\u0900-\u097F]", page_text))
        looks_like_factcheck = (
            re.search(r"fact|check|तथ्य", page_title, re.I) is not None
            and len(internal_links) > 0.3 * max(len(hrefs), 1)
        )
        logger.info("Homepage title: '%s' | links: %d (internal %d) | Devanagari chars: %d",
                    page_title[:80], len(hrefs), len(internal_links), devanagari_chars)
        if not looks_like_factcheck:
            logger.warning("Domain no longer serves fact-check content (unrelated site/link farm) — skipping %s", SOURCE)
            site_stats["status"] = "NOT_FACTCHECK_CONTENT"
            site_failures.append({"source": SOURCE, "url": BASE_URL + "/", "status_code": resp.status_code,
                                  "error": "Domain serves unrelated content; no fact-check articles found"})
        else:
            logger.warning("Fact-check content detected — inspect the site and add selectors here before scraping")
            site_stats["status"] = "REACHABLE_NOT_IMPLEMENTED"
    except requests.RequestException as exc:
        logger.error("Request failed: %s", exc)
        site_failures.append({"source": SOURCE, "url": BASE_URL + "/",
                              "status_code": getattr(getattr(exc, "response", None), "status_code", None),
                              "error": str(exc)})
        site_stats["status"] = "HTTP_FAILURE"

records_by_source[SOURCE] = site_records
failures_by_source[SOURCE] = site_failures
stats_by_source[SOURCE] = site_stats
logger.info("%s finished | %s", SOURCE, site_stats)

2026-11-22 15:27:02 | INFO | ============================================================
2026-11-22 15:27:02 | INFO | Starting South Asia Check scraper
2026-11-22 15:27:02 | INFO | Fetching robots.txt: https://southasiacheck.org/robots.txt
2026-11-22 15:27:02 | INFO | HTTP 200
2026-11-22 15:27:04 | INFO | Fetching page: https://southasiacheck.org/
2026-11-22 15:27:04 | INFO | HTTP 200
2026-11-22 15:27:04 | INFO | Homepage title: 'Sudjaad168: สมัครสล็อต API แท้ 100% รองรับวอเลท ปี 2026' | links: 393 (internal 0) | Devanagari chars: 0
2026-11-22 15:27:04 | WARNING | Domain no longer serves fact-check content (unrelated site/link farm) — skipping South Asia Check
2026-11-22 15:27:04 | INFO | South Asia Check finished | {'source': 'South Asia Check', 'pages_discovered': 0, 'urls_found': 0, 'successful': 0, 'skipped': 0, 'failed': 0, 'duplicate_urls': 0, 'status': 'NOT_FACTCHECK_CONTENT'}


In [0]:
%python
# ============================================================
# STEP 12 — Combine all sources and deduplicate
# ============================================================
all_records = []
for src_name in records_by_source:
    all_records.extend(records_by_source[src_name])
combined_df = pd.DataFrame(all_records, columns=OUTPUT_COLUMNS)
logger.info("Combined raw records: %d", len(combined_df))

# Normalised URL (scheme/host lower-case, no query/fragment/trailing slash)
normalised_urls = []
for u in combined_df["article_url"]:
    if isinstance(u, str) and u:
        pu = urlparse(u.strip())
        normalised_urls.append(urlunparse((pu.scheme.lower(), pu.netloc.lower(), pu.path.rstrip("/"), "", "", "")))
    else:
        normalised_urls.append(None)
combined_df["_norm_url"] = normalised_urls
# Canonical key: normalised URL; records without a URL (TechPana) fall back to source + article_id
combined_df["_dedup_key"] = [nu if nu else f"{s}|{aid}" for nu, s, aid in
                             zip(combined_df["_norm_url"], combined_df["source"], combined_df["article_id"])]

before_dedup = len(combined_df)
dup_mask = combined_df.duplicated(subset=["_dedup_key"], keep="first")
for dup_key in combined_df.loc[dup_mask, "_dedup_key"]:
    logger.warning("Duplicate URL removed: %s", dup_key)
final_df = combined_df.loc[~dup_mask].copy()
DUPLICATES_REMOVED = before_dedup - len(final_df)
logger.info("Duplicates removed (same canonical URL): %d", DUPLICATES_REMOVED)

# Possible duplicates (flag only — different URLs are kept as separate articles)
possible_dups = final_df[final_df.duplicated(subset=["source", "title", "published_date"], keep=False)]
logger.info("Possible duplicates (same source + title + published_date, different URL): %d rows", len(possible_dups))
cross_source_titles = final_df.groupby("title")["source"].nunique()
cross_source_titles = cross_source_titles[cross_source_titles > 1]
logger.info("Titles published by more than one source (kept, e.g. network republishing): %d", len(cross_source_titles))

final_df = final_df.drop(columns=["_norm_url", "_dedup_key"]).reset_index(drop=True)
logger.info("Final combined record count: %d", len(final_df))
for src_name, src_count in final_df["source"].value_counts().items():
    logger.info("Final records | %-22s %d", src_name, src_count)

2026-11-22 15:27:43 | INFO | Combined raw records: 1244
2026-11-22 15:27:43 | INFO | Duplicates removed (same canonical URL): 0
2026-11-22 15:27:43 | INFO | Possible duplicates (same source + title + published_date, different URL): 0 rows
2026-11-22 15:27:43 | INFO | Titles published by more than one source (kept, e.g. network republishing): 357
2026-11-22 15:27:43 | INFO | Final combined record count: 1244
2026-11-22 15:27:43 | INFO | Final records | Nepal Factcheck        659
2026-11-22 15:27:43 | INFO | Final records | Karnali Factcheck      357
2026-11-22 15:27:43 | INFO | Final records | TechPana Fact Check    115
2026-11-22 15:27:43 | INFO | Final records | Sachilo                84
2026-11-22 15:27:43 | INFO | Final records | Lumbini Factcheck      29


In [0]:
%python
# ============================================================
# STEP 13 — In-memory quality checks (before writing CSV)
# ============================================================
non_null_urls = final_df["article_url"].dropna()
invalid_urls = [u for u in non_null_urls if not (urlparse(u).scheme in ("http", "https") and urlparse(u).netloc)]

quality_report = pd.DataFrame([
    ("total records", len(final_df)),
    ("missing title", int(final_df["title"].isna().sum())),
    ("missing URL (TechPana publishes none)", int(final_df["article_url"].isna().sum())),
    ("missing claim", int(final_df["claim"].isna().sum())),
    ("missing verdict", int(final_df["verdict"].isna().sum())),
    ("empty article_text", int(final_df["article_text"].isna().sum())),
    ("duplicate URLs", int(non_null_urls.duplicated().sum())),
    ("duplicate title + source", int(final_df.duplicated(subset=["source", "title"]).sum())),
    ("invalid URLs", len(invalid_urls)),
], columns=["check", "value"])
logger.info("Quality report:\n%s", quality_report.to_string(index=False))
display(quality_report)

missing_by_source = final_df[["title", "claim", "verdict", "article_text", "author"]] \
    .isna().groupby(final_df["source"]).sum()
missing_by_source.insert(0, "records", final_df["source"].value_counts())
logger.info("Missing values by source:\n%s", missing_by_source.to_string())
display(missing_by_source.reset_index())

records_by_source_df = final_df["source"].value_counts().rename_axis("source").reset_index(name="records")
records_by_verdict_df = final_df.groupby(["source", "verdict"], dropna=False).size().reset_index(name="records") \
    .sort_values(["source", "records"], ascending=[True, False])
records_by_language_df = final_df["language"].value_counts(dropna=False).rename_axis("language").reset_index(name="records")
years = final_df["published_date"].astype("string").str.extract(r"^(\d{4})")[0]
records_by_year_df = pd.crosstab(years.fillna("unknown"), final_df["source"]).reset_index().rename(columns={0: "year"})

display(records_by_source_df)
display(records_by_verdict_df)
display(records_by_language_df)
display(records_by_year_df)

2026-11-22 15:27:50 | INFO | Quality report:
                                check  value
                        total records   1244
                        missing title      0
missing URL (TechPana publishes none)    115
                        missing claim    249
                      missing verdict      0
                   empty article_text    115
                       duplicate URLs      0
             duplicate title + source      1
                         invalid URLs      0


check,value
total records,1244
missing title,0
missing URL (TechPana publishes none),115
missing claim,249
missing verdict,0
empty article_text,115
duplicate URLs,0
duplicate title + source,1
invalid URLs,0


2026-11-22 15:27:51 | INFO | Missing values by source:
                     records  title  claim  verdict  article_text  author
source                                                                   
Karnali Factcheck        357      0     53        0             0     343
Lumbini Factcheck         29      0      7        0             0      26
Nepal Factcheck          659      0     55        0             0       0
Sachilo                   84      0     19        0             0       0
TechPana Fact Check      115      0    115        0           115       0


source,records,title,claim,verdict,article_text,author
Karnali Factcheck,357,0,53,0,0,343
Lumbini Factcheck,29,0,7,0,0,26
Nepal Factcheck,659,0,55,0,0,0
Sachilo,84,0,19,0,0,0
TechPana Fact Check,115,0,115,0,115,0


source,records
Nepal Factcheck,659
Karnali Factcheck,357
TechPana Fact Check,115
Sachilo,84
Lumbini Factcheck,29


source,verdict,records
Karnali Factcheck,मिथ्या सूचना,197
Karnali Factcheck,भ्रामक सूचना,136
Karnali Factcheck,अपुष्ट सूचना,15
Karnali Factcheck,सही सूचना,7
Karnali Factcheck,"अपुष्ट सूचना, भ्रामक सूचना",1
Karnali Factcheck,"अपुष्ट सूचना, मिथ्या सूचना",1
Lumbini Factcheck,मिथ्या सूचना,18
Lumbini Factcheck,भ्रामक सूचना,10
Lumbini Factcheck,सही सूचना,1
Nepal Factcheck,भ्रामक सूचना,343


language,records
ne,1152
mixed,48
en,44


year,Karnali Factcheck,Lumbini Factcheck,Nepal Factcheck,Sachilo,TechPana Fact Check
2020,98,9,99,0,0
2021,44,1,44,0,0
2022,54,16,54,0,0
2023,54,2,54,0,0
2024,102,1,102,0,115
2025,5,0,91,0,0
2026,0,0,215,84,0


In [0]:
%python
# ============================================================
# STEP 14 — Write the combined dataset as ONE UTF-8 CSV next to this notebook
# ============================================================
# pandas writes a single file directly (no Spark part-files, no tables).
RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
CSV_PATH = os.path.join(OUTPUT_DIR, CSV_FILENAME)
if os.path.exists(CSV_PATH):
    CSV_PATH = os.path.join(OUTPUT_DIR, CSV_FILENAME.replace(".csv", f"_{RUN_TS}.csv"))
    logger.warning("%s already exists — not overwriting; writing timestamped file instead", CSV_FILENAME)
logger.info("Writing CSV: %s", CSV_PATH)
final_df.to_csv(CSV_PATH, index=False, encoding="utf-8", quoting=csv.QUOTE_ALL, doublequote=True)
CSV_SIZE_BYTES = os.path.getsize(CSV_PATH)
logger.info("CSV written | %d rows | %.2f MB", len(final_df), CSV_SIZE_BYTES / 1024 / 1024)

# Read back to verify row count, columns and Nepali Unicode round-trip
check_df = pd.read_csv(CSV_PATH, encoding="utf-8", dtype=str, keep_default_na=False)
assert len(check_df) == len(final_df), f"Row count mismatch: {len(check_df)} vs {len(final_df)}"
assert list(check_df.columns) == OUTPUT_COLUMNS, "Column mismatch after read-back"
roundtrip_ok = check_df["title"].tolist() == final_df["title"].fillna("").tolist()
devanagari_rows = int(check_df["title"].str.contains(r"[\u0900-\u097F]", regex=True).sum())
logger.info("Read-back OK | rows=%d | titles identical=%s | rows with Devanagari titles=%d",
            len(check_df), roundtrip_ok, devanagari_rows)

# Failure log (optional) — also CSV only
all_failures = []
for src_name in failures_by_source:
    all_failures.extend(failures_by_source[src_name])
failures_df = pd.DataFrame(all_failures, columns=["source", "url", "status_code", "error"])
FAILURES_CSV_PATH = None
if len(failures_df):
    FAILURES_CSV_PATH = os.path.join(OUTPUT_DIR, FAILURES_CSV_FILENAME)
    if os.path.exists(FAILURES_CSV_PATH):
        FAILURES_CSV_PATH = os.path.join(OUTPUT_DIR, FAILURES_CSV_FILENAME.replace(".csv", f"_{RUN_TS}.csv"))
    failures_df.to_csv(FAILURES_CSV_PATH, index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)
    logger.info("Failure log written: %s (%d rows)", FAILURES_CSV_PATH, len(failures_df))
display(failures_df)

2026-11-22 15:27:57 | INFO | Writing CSV: /Workspace/Users/ashresth.ycotek@cpaxtra.co.th/Testss/nepali_fact_check_dataset.csv
2026-11-22 15:27:58 | INFO | CSV written | 1244 rows | 19.85 MB
2026-11-22 15:27:58 | INFO | Read-back OK | rows=1244 | titles identical=True | rows with Devanagari titles=1215
2026-11-22 15:27:59 | INFO | Failure log written: /Workspace/Users/ashresth.ycotek@cpaxtra.co.th/Testss/nepali_fact_check_failures.csv (3 rows)


source,url,status_code,error
The Ruju,https://www.theruju.com/robots.txt,403,Site returns HTTP 403 (access restricted); not scraped
NepalCheck,https://nepalcheck.org/robots.txt,200,robots.txt Disallow: / — not scraped
South Asia Check,https://southasiacheck.org/,200,Domain serves unrelated content; no fact-check articles found


In [0]:
%python
# ============================================================
# STEP 15 — Final summary
# ============================================================
summary_rows = []
for src_name in stats_by_source:
    st = stats_by_source[src_name]
    summary_rows.append({
        "Source": src_name, "Articles Found": st["urls_found"], "Successful": st["successful"],
        "Failed": st["failed"], "Skipped": st["skipped"],
        "Final Records": int((final_df["source"] == src_name).sum()), "Status": st["status"],
    })
summary_df = pd.DataFrame(summary_rows)

lines = ["=" * 78, "NEPALI FACT-CHECK DATASET SCRAPING COMPLETE", "=" * 78, "",
         f"{'Source':<22}{'Found':>8}{'Success':>9}{'Failed':>8}{'Skipped':>9}  Status"]
for row in summary_rows:
    lines.append(f"{row['Source']:<22}{row['Articles Found']:>8}{row['Successful']:>9}{row['Failed']:>8}"
                 f"{row['Skipped']:>9}  {row['Status']}")
lines += ["", f"{'TOTAL':<22}{summary_df['Articles Found'].sum():>8}{summary_df['Successful'].sum():>9}"
              f"{summary_df['Failed'].sum():>8}{summary_df['Skipped'].sum():>9}",
          "", f"Duplicates removed: {DUPLICATES_REMOVED}",
          f"Final dataset records: {len(final_df)}",
          "", "CSV:", CSV_PATH, "", "File size:", f"{CSV_SIZE_BYTES / 1024 / 1024:.2f} MB ({CSV_SIZE_BYTES:,} bytes)",
          "", f"Failure log: {FAILURES_CSV_PATH}",
          "", f"Status: {'SUCCESS' if len(check_df) == len(final_df) and len(final_df) > 0 else 'FAILED'}",
          "=" * 78]
for ln in lines:
    logger.info(ln)

# First 10 rows for validation (long text fields truncated for display only)
preview_df = final_df.head(10).copy()
for col in ["claim", "description", "article_text", "evidence", "sources", "tags"]:
    preview_df[col] = preview_df[col].astype("string").str.slice(0, 120)
display(preview_df)

2026-11-22 15:28:04 | INFO | ==============================================================================
2026-11-22 15:28:04 | INFO | NEPALI FACT-CHECK DATASET SCRAPING COMPLETE
2026-11-22 15:28:04 | INFO | ==============================================================================
2026-11-22 15:28:04 | INFO | 
2026-11-22 15:28:04 | INFO | Source                   Found  Success  Failed  Skipped  Status
2026-11-22 15:28:04 | INFO | Nepal Factcheck            659      659       0        0  OK
2026-11-22 15:28:04 | INFO | The Ruju                     0        0       0        0  ACCESS_FORBIDDEN_HTTP_403
2026-11-22 15:28:04 | INFO | TechPana Fact Check        115      115       0        0  OK_LIMITED_FIELDS
2026-11-22 15:28:04 | INFO | Karnali Factcheck          357      357       0        0  OK
2026-11-22 15:28:04 | INFO | Lumbini Factcheck           29       29       0        0  OK
2026-11-22 15:28:04 | INFO | Sachilo                     84       84       0        0  OK
2026-11-2

source,article_url,article_id,title,claim,verdict,claimant,claim_date,fact_check_date,published_date,author,category,language,description,article_text,evidence,sources,correction,image_url,tags,scraped_at
Nepal Factcheck,https://nepalfactcheck.org/2026/11/ai-generated-false-videos-of-prime-minister-balen-and-other-top-officials/,12006,"नेताको अनुहार, एआईको बोली : फेसबुकमा कसले कसरी हामीलाई झुक्क्याउँदै छन् ?",तिहार सुरु हुनुभन्दा केही दिनअघि २०८३ सालको कार्तिक १७ गते (नोभेम्बर ३) को दिन अर्थमन्त्री स्वर्णिम वाग्लेको दाबी गर्दै,मिथ्या सूचना,null,null,null,2026-11-18T10:06:52,टेकमान शाक्य,एक्सप्लेनर,ne,प्रधानमन्त्री र मन्त्रीसहित राजनीतिक दलका शीर्ष नेता एवम् उच्च पदस्थ व्यक्तिको अनुहार मिल्ने गरी एआई निर्मित तस्बिर एवम्,प्रधानमन्त्री र मन्त्रीसहित राजनीतिक दलका शीर्ष नेता एवम् उच्च पदस्थ व्यक्तिको अनुहार मिल्ने गरी एआई निर्मित तस्बिर एवम्,null,https://www.facebook.com/share/r/18pkYgo6Kf/ | https://www.facebook.com/reel/1793847228586078 | https://www.facebook.com,null,https://i0.wp.com/nepalfactcheck.org/wp-content/uploads/2026/11/Explainer-1.png?fit=1280%2C720&ssl=1,"AI Fake Videos, AI Generated Videos, AI manipulation, Artificial Intelligence, balen shah, Balendra Shah, Deepfake Video",2026-11-22T15:14:10.443013+00:00
Nepal Factcheck,https://nepalfactcheck.org/2026/10/budhi-gandaki-flood-viral-photo-from-chile-misleading-claim/,11925,"‘बुढीगण्डकीको बाढीमा गाडीमाथि चढेर ज्यान बचाएको’ दाबी भ्रामक, भाइरल फोटो चिलीको",फेसबुकमा १७ हजार फलोअर्स रहेका प्रयोगकर्ता यज्ञ खड्का ले अक्टोबर ११ (आइतबार) दिउँसो १:४३ बजे एउटा फोटो अपलोड गरे। बुढीगण,भ्रामक सूचना,null,null,null,2026-10-14T11:48:30,सुरज घिमिरे,"जलवायु र वातावरण, वागमती",ne,अहिले सामाजिक सञ्जालमा “बुढीगण्डकीमा भीषण बाढी आएपछि एक जना व्यक्ति गाडीमाथि चढेर ज्यान बचाए” भन्दै एउटा फोटो भाइरल भइरह,अहिले सामाजिक सञ्जालमा “बुढीगण्डकीमा भीषण बाढी आएपछि एक जना व्यक्ति गाडीमाथि चढेर ज्यान बचाए” भन्दै एउटा फोटो भाइरल भइरह,के हो तथ्य ? तथ्य जाँचका क्रममा हामीले सुरुमा ‘बुढीगण्डकी बाढी’सम्बन्धी किवर्डहरू राखेर गुगलमा सर्च गर्‍यौँ । तर कुनै पन,https://www.facebook.com/juzing.king | https://www.facebook.com/share/p/1VPt9s1MGF/ | https://archive.ph/qM5YH | https:/,null,https://i0.wp.com/nepalfactcheck.org/wp-content/uploads/2026/10/Budigandaki_Fact-Check.png?fit=1280%2C720&ssl=1,"Budhi Gandaki flood, Budhi Gandaki flood fact check, Chile flash flood 2026, Chile flood, CNN Chile flood video, Disinfo",2026-11-22T15:14:12.562838+00:00
Nepal Factcheck,https://nepalfactcheck.org/2026/10/dashain-heavy-rainfall-viral-video-misleading-claim-nepal/,11904,"दशैँको टीकाको भोलिपल्ट भारी वर्षा हुने खबर भ्रामक, पुरानो अडिओ जोडेर भ्रम फैलाइयो",फेसबुकमा १५ हजार फलोअर्स रहेकी विष्णु घिमिरे नामक प्रयोगकर्ता ले अक्टोबर ९ (असोज २३ गते) एउटा भिडिओ अपलोड गरिन् । बाढीको,भ्रामक सूचना,null,null,null,2026-10-12T15:14:36,सुरज घिमिरे,जलवायु र वातावरण,ne,सामाजिक सञ्जालमा दशैँको टीकाको भोलिपल्टदेखि नेपालमा भारीदेखि अति भारी वर्षा हुने सम्भावना रहेको भन्दै सामग्रीहरू भाइरल ब,सामाजिक सञ्जालमा दशैँको टीकाको भोलिपल्टदेखि नेपालमा भारीदेखि अति भारी वर्षा हुने सम्भावना रहेको भन्दै सामग्रीहरू भाइरल ब,के हो तथ्य ? तथ्य जाँचका क्रममा हामीले सुरुमा दशैँको समयमा वर्षा हुने सम्भावना सम्बन्धी आधिकारिक समाचारहरू खोज्यौँ । तर,https://www.facebook.com/bishnu.ghimire.3956690 | https://www.facebook.com/share/r/1CaHEVUbx3/ | https://archive.ph/vEMf,null,https://i0.wp.com/nepalfactcheck.org/wp-content/uploads/2026/10/Nana_Fact-Check-1-1.png?fit=1280%2C720&ssl=1,"Dashain 2026, Dashain Weather Forecast, Department of Hydrology and Meteorology, Extreme Rainfall Claim, fact-check Nepa",2026-11-22T15:14:14.612305+00:00
Nepal Factcheck,https://nepalfactcheck.org/2026/10/claim-that-41-nepalis-converted-to-islam-after-rasuwa-flood-is-misleading/,11881,रसुवाको बाढीपछि ४१ नेपालीले धर्म परिवर्तन गरेको दाबीसहितको भिडिओ भ्रामक,१ लाख २७ हजार फलोअर्स रहेको ‘Bedika KC’ नामक फेसबुक प्रयोगकर्ताले सन् २०२६ अक्टोबर १० का दिन बेलुका ६ बजेर १८ मिनेटमा ४,भ्रामक सूचना,null,null,null,2026-10-11T17:31:48,टेकमान शाक्य,null,ne,रसुवाको भोटेकोशी बाढीपछि सनातन धर्म मान्ने ४१ ज

In [0]:
SELECT * FROM read_files('file:/Workspace/Users/ashresth.ycotek@cpaxtra.co.th/Testss/nepali_fact_check_dataset.csv', format => 'csv', header => 'true', multiLine => 'true', escape => '"')

source article_url article_id title claim verdict claimant claim_date fact_check_date published_date author category language description article_text evidence sources correction image_url tags scraped_at _rescued_data Nepal Factcheck https://nepalfactcheck.org/2026/11/ai-generated-false-videos-of-prime-minister-balen-and-other-top-officials/ 12006 नेताको अनुहार, एआईको बोली : फेसबुकमा कसले कसरी हामीलाई झुक्क्याउँदै छन् ? तिहार सुरु हुनुभन्दा केही दिनअघि २०८३ सालको कार्तिक १७ गते (नोभेम्बर ३) को दिन अर्थमन्त्री स्वर्णिम वाग्लेको दाबी गर्दै २० सेकेन्डको भिडिओ क्लिप भाइरल भयो । Gangu Chhetri नामक फेसबुक पेजबाट सार्वजनिक भिडिओमा सरकारले तिहारमा बोनस बाँड्न सुरु गरेको दाबी गरिएको छ ।
भिडिओमा ४ हजारभन्दा धेरैपटक रियाक्ट, २ हजार ७ सय भन्दा धेरै पटक कमेन्ट र ७ सय ४६ भन्दान धेरैपटक सेयर गरिएको छ । कमेन्ट गर्ने अधिकांशले भिडिओलाई वास्तविक ठानेर आफ्नो विचार व्यक्त गरिरहेका छन् । सरकारलाई धन्यवाद दिईरहेका छन् ।
यो भिडिओमा सुनिएको आवाज अर्थमन्त्री वाग्लेको वास्तविक आवाजसँग मेल खाएको सुनिँदैन । त्यसैले हामीले दृश्य र अडिओ एआई निर्मित हो कि होइन भनेर जाँच गर्‍यौँ । एआई डिटेक्शन टुल HIVE MODERATION ले दिएको नतिजा अनुसार भिडिओ र अडिओ दुवै एआई निर्मित एवम् सम्पादित भएको भेटियो ।
यो भिडिओ भाइरल भएपछि यहि पेजमार्फत कार्तिक १८ गते (नोभेम्बर ४) को दिन कानुनमन्त्री सोबिता गौतमको अर्को भिडिओ क्लिप सार्वजनिक गरियो । भिडिओमा कानुनमन्त्री गौतमले संसद् बैठकमा सम्बोधन गर्दै तिहारमा बोनस पाउने कुरा सत्य भएको दाबी गरिएको सुनिन्छ ।
भिडिओको शीर्षकमा बोनस रकम २० हजार रुपैयाँ पाउने भनेर दाबी गरिएको छ । अधिकांशले कमेन्टमा मन्त्री गौतम र सरकारलाई धन्यवाद दिएका छन् ।
७ हजारभन्दा धेरै पटक रियाक्ट गरिएको, ३ हजारभन्दा धेरैपटक कमेन्ट र ४ सय ८६ पटकभन्दा धेरैपटक सेयर गरिएको कानुनमन्त्री गौतमको यो भिडिओ पनि एआई निर्मित पुष्टि हुन्छ । सिन्थआईडी डटकममार्बाफत जाँच गरेर हेर्दा भिडिओ गुगल एआईमार्फत सम्पादन एवम् निर्माण गरेको भेटियो ।
Gangu Chhetri नामक फेसबुक पेजमार्फत यस्ता थुप्रै यस्ता भिडिओ क्लिप सार्वजनिक भएका छन् । बृद्धभत्ता ४ गुणाले बढेर ८ हजार पुगेको भनेर रास्ट्रिय स्वतन्त्र पार्टी रास्वपाका सभापति रवि लामिछानेले बोलेको, अब खोरमा जाने पालो केपी ओलीको लेखिएको पोस्टर गृहमन्त्री सुधन गुरुङले बोकेको लगायत दाबीसहितको तस्बिर र भिडिओ यो पेजमार्फत निरन्तर सार्वजनिक भइरहेका छन् ।
निरन्तर पेजमार्फत मिथ्या सामग्री पोस्ट भएको भेटिएसँगै हामीले यो पेज कहाँबाट सञ्चालित छ भनेर थप विवरण खोजी गर्‍यौँ । २४ हजार फलोअर्स रहेको Gangu Chhetri नामक यो पेज सन् २०२५ को सेप्टेम्बर १७ मा खोलिएको र भारतको लोकेशनबाट सञ्चालित रहेको पेज ट्रान्सपरेन्सीमा देखियो । पेजमा ईमेल ठेगाना, सम्पर्क नम्बर लगायत आधिकारिक विवरण सार्वजनिक गरिएको छैन । मिथ्या सूचना null null null 2026-11-18T10:06:52+07:00 टेकमान शाक्य एक्सप्लेनर ne प्रधानमन्त्री र मन्त्रीसहित राजनीतिक दलका शीर्ष नेता एवम् उच्च पदस्थ व्यक्तिको अनुहार मिल्ने गरी एआई निर्मित तस्बिर एवम् भिडिओ सामग्री सामाजिक सञ्जालमा भाइरल भइरहेका छन् । इमेल ठेगाना, सम्पर्क नम्बर या अरू वास्तविक पहिचान नखुलाई चलाइएका फेसबुक पेजमार्फत यस्ता सामग्री सार्वजनिक भएका हुन् । नेताको अनुहार प्रयोग गरेर एआईमार्फत आवाज र दृश्य निर्माण गरिएका भिडिओलाई […] प्रधानमन्त्री र मन्त्रीसहित राजनीतिक दलका शीर्ष नेता एवम् उच्च पदस्थ व्यक्तिको अनुहार मिल्ने गरी एआई निर्मित तस्बिर एवम् भिडिओ सामग्री सामाजिक सञ्जालमा भाइरल भइरहेका छन् । इमेल ठेगाना, सम्पर्क नम्बर या अरू वास्तविक पहिचान नखुलाई चलाइएका फेसबुक पेजमार्फत यस्ता सामग्री सार्वजनिक भएका हुन् ।

नेताको अनुहार प्रयोग गरेर एआईमार्फत आवाज र दृश्य निर्माण गरिएका भिडिओलाई धेरैले वास्तविक ठानेर प्रतिक्रिया जनाइरहेका छन्।अर्थात् सामाजिक सञ्जाल प्रयोगकर्ता यस्ता सामग्रीबाट झुक्किएका छन् ।

तिहारको बोनसबारे अर्थमन्त्री र कानुनमन्त्रीको मिथ्या भिडिओ

तिहार सुरु हुनुभन्दा केही दिनअघि २०८३ सालको कार्तिक १७ गते (नोभेम्बर ३) को दिन अर्थमन्त्री स्वर्णिम वाग्लेको दाबी गर्दै २० सेकेन्डको भिडिओ क्लिप भाइरल भयो । Gangu Chhetri नामक फेसबुक पेजबाट सार्वजनिक भिडिओमा सरकारले तिहारमा बोनस बाँड्न सुरु गरेको दाबी गरिएको छ ।

भिडिओमा ४ हजारभन्दा धेरैपटक रियाक्ट, २ हजार ७ सय भन्दा धेरै पटक कमेन्ट र ७ सय ४६ भन्दान धेरैपटक सेयर गरिएको छ । कमेन्ट गर्ने अधिकांशले भिडिओलाई वास्तविक ठानेर आफ्नो विचार व्यक्त गरिरहेका छन् । सरकारलाई धन्यवाद दिईरहेका छन् ।

यो भिडिओमा 